# Preprocessing of the RTS data to prepare for the Issue Index creation

In [1]:
import os
import json
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
from tqdm import tqdm
import csv
from ast import literal_eval
from collections import Counter

### Directory Structure: `/mnt/project_impresso/original/RTS`

The RTS directory contains radio content organized by program/show names. The filesystem structure is organized as follows:

```
/mnt/project_impresso/original/RTS/
│
├── News Programs (Journaux)
│   ├── j_mat/                    (morning news)  
│   │   ├── audio                       (all PM3 audio files)
│   │   ├── stt                         (all stt and xml files containing the text related to each audio file)
│   │   ├── ExpXml-20251106-122951.xml  (xml files with the metadata relating the MP3 audio and XML files for each listing)
│   │   ...  
│   │   └── ExpXml-20251113-171933.xml                   
│   ├── j_midi/                   (midday news)
│   ├── j13h/                     (1 PM news)
│   ├── j13h2/                    (1 PM news variant)
│   ├── j_soir/                   (evening news)
│   └── j_nuit/                   (night news)
│
...
│
└── Other Programs
    ├── petitdej/                 (breakfast)
    ├── ana_media/                (media analysis)
    ├── geneve_info/              (Geneva information)
    └── enquest/                  (enquête/investigation)
```

Each directory contains media files (audio recordings and associated metadata) for that specific radio program.

In [2]:
base_dir = "/mnt/project_impresso/original/RTS"

audios_subdir = 'audio'
asr_subdir = 'stt'
metadata_file_start = "ExpXml"

## Process an example of metadata xml file to extract the contents

In [4]:
example_program = "causerie_uni"

example_program_dir = os.path.join(base_dir, example_program)

ex_meta_files = [os.path.join(example_program_dir,f) for f in os.listdir(example_program_dir) if f.startswith(metadata_file_start)]
ex_meta_files

['/mnt/project_impresso/original/RTS/causerie_uni/ExpXml-20251029-131700.xml',
 '/mnt/project_impresso/original/RTS/causerie_uni/ExpXml-20251029-132007.xml']

In [25]:
with open(ex_meta_files[0], "r", encoding="utf-8") as f:
    raw_xml = f.read()

xml_doc = BeautifulSoup(raw_xml, "xml")
xml_doc

<?xml version="1.0" encoding="utf-8"?>
<DOCUMENTS><DOCUMENT><CLSID>{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}</CLSID><OID>{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}</OID><LOGIN/><TITLE>Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg</TITLE><HIERARCHY OID="{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}" current="*" depth="---" title="Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg"/><SEQUENCE>1</SEQUENCE><BROADCAST>Causerie universitaire</BROADCAST><DOCUMENTTYPE>Parl?</DOCUMENTTYPE><GEOGRAPHICALDESCRIPTORS><GEOGRAPHICALDESCRIPTOR>Pologne</GEOGRAPHICALDESCRIPTOR></GEOGRAPHICALDESCRIPTORS><HIERARCHYLEVEL>Sujet</HIERARCHYLEVEL><MODIFIEDBY>albrecjo</MODIFIEDBY><MODIFIEDON>25.05.2022 03:56:53</MODIFIEDON><HISTORY>Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B</HISTORY><PARTICIPANTS><PARTICIPANT><NAME>Cros, Edouard</NAME><FUNCTION>Conf?rencier/e</FUNCTION><RO

In [6]:
docs = xml_doc.find_all("DOCUMENT")
docs 

[<DOCUMENT><CLSID>{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}</CLSID><OID>{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}</OID><LOGIN/><TITLE>Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg</TITLE><HIERARCHY OID="{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}" current="*" depth="---" title="Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg"/><SEQUENCE>1</SEQUENCE><BROADCAST>Causerie universitaire</BROADCAST><DOCUMENTTYPE>Parl?</DOCUMENTTYPE><GEOGRAPHICALDESCRIPTORS><GEOGRAPHICALDESCRIPTOR>Pologne</GEOGRAPHICALDESCRIPTOR></GEOGRAPHICALDESCRIPTORS><HIERARCHYLEVEL>Sujet</HIERARCHYLEVEL><MODIFIEDBY>albrecjo</MODIFIEDBY><MODIFIEDON>25.05.2022 03:56:53</MODIFIEDON><HISTORY>Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B</HISTORY><PARTICIPANTS><PARTICIPANT><NAME>Cros, Edouard</NAME><FUNCTION>Conf?rencier/e</FUNCTION><ROLE>privat-docent ? l'Universit? de Fribourg</ROLE

In [7]:
for child in docs[0].find_all(recursive=False):
    print(f"name: {child.name}")
    print(f"attrs: {child.attrs}")
    print(f"text: {child.get_text(strip=True)}")

name: CLSID
attrs: {}
text: {D2593F4E-C887-4E48-8982-5BD08BA4DAE0}
name: OID
attrs: {}
text: {3267F04D-657F-4DCB-BCB0-44B2B6C2682D}
name: LOGIN
attrs: {}
text: 
name: TITLE
attrs: {}
text: Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg
name: HIERARCHY
attrs: {'title': "Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg", 'OID': '{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}', 'depth': '---', 'current': '*'}
text: 
name: SEQUENCE
attrs: {}
text: 1
name: BROADCAST
attrs: {}
text: Causerie universitaire
name: DOCUMENTTYPE
attrs: {}
text: Parl?
name: GEOGRAPHICALDESCRIPTORS
attrs: {}
text: Pologne
name: HIERARCHYLEVEL
attrs: {}
text: Sujet
name: MODIFIEDBY
attrs: {}
text: albrecjo
name: MODIFIEDON
attrs: {}
text: 25.05.2022 03:56:53
name: HISTORY
attrs: {}
text: Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B
name: PARTICIPANTS
attrs: {}
text: Cros, 

In [12]:
# define more intuitive names
simple_fields_renaming = {
    "CLSID": "cls_ID",
    "OID": "OID",
    "LOGIN": "login",
    "TITLE": "broadcast_episode_title",
    "SEQUENCE": "sequence", 
    "BROADCAST": "broadcast_program_name",
    "DOCUMENTTYPE": "document_type",
    "HIERARCHYLEVEL": "hierarchy_level",
    "MODIFIEDBY": "modified_by",
    "MODIFIEDON": "modified_on",
    "HISTORY": "physical_support_history",
    "PRODUCTIONTYPE": "production_type",
    "RECORDINGPLACE": "recording_place",
    "RIGHTSNOTES": "rights_notes",
    "RIGHTSSTATUS": "rights_status",
    "SERIESTITLE": "series_title",
    "SUMMARY": "content_summary",
    "WORKFLOWSTATUS": "workflow_status",
    "ASSEMBLYSTATUS": "assembly_status",
    "LIVE": "live",
    "MODULATIONTYPE": "modulation_type",
    "WORKDURATION": "work_duration",
    "WORKDURATIONCOMPL": "work_duration_compl",
}

list_fields_renaming = {
    'GEOGRAPHICALDESCRIPTORS': 'geographical_descriptors',
    'PERSONDESCRIPTORS': 'person_descriptors',
    'THEMATICALDESCRIPTORS': 'thematical_descriptors',
    'RIGHTSUSAGEPOSSIBILITIES': 'rights_usage_possibilities',
    'PROGRAMMES': 'radio_channels',
    'SUBDOMAINS': 'subdomains',
    'RECORDINGDATES': 'recording_dates',
    'FIRSTBROADCASTDATES': 'first_broadcast_dates'
}

support_keys = ['spt_clsid', 'spt_oid',
                'spt_title',
                'spt_isdigital',
                'spt_filename',
                'spt_cataloguing_status',
                'spt_source',
                'spt_unit_duration']

doc_keys = ["alias", "date_str", "stt_filename", "mp3_filenames", "stripped_OID", "exact_date", "broadcast_date"] + list(simple_fields_renaming.values()) + list(list_fields_renaming.values()) + ["supports", "spt_filenames", "participants"]

In [13]:
# Parse DOCUMENT elements into structured dictionaries
def parse_docs_in_xml(xml_doc, program, all_alias_stt, all_alias_audios, doc_keys=doc_keys, simple_fields_map=simple_fields_renaming, list_fields_map=list_fields_renaming):
    """
    Extract all DOCUMENT elements from XML into a list of dictionaries.
    Handles nested structures: PARTICIPANTS, GEOGRAPHICALDESCRIPTORS, 
    THEMATICALDESCRIPTORS, SUPPORTS, RECORDINGDATES, etc.
    """
    documents = []
    
    # Find all DOCUMENT elements
    doc_elements = xml_doc.find_all('DOCUMENT')
    skipped = 0
    
    print(f"\nStarting extracting {len(doc_elements)} documents for program {program}")
    for doc_idx, doc_elem in enumerate(doc_elements):
        doc_dict = {
            "alias": program,
            "date_str": None,
            "stt_filename": None,
            "mp3_filenames": None
        }
        
        # Extract simple text fields
        
        
        for og_field, renamed_field in simple_fields_map.items():
            elem = doc_elem.find(og_field)
            if elem:
                text = elem.get_text(strip=True)
                doc_dict[renamed_field] = text
                if og_field=='OID':
                    # the text is actually "{oid}", remove start and end brackets
                    doc_dict['stripped_OID'] = text[1:-1] if text else None
            else:
                doc_dict[renamed_field] = None
        
        if "stripped_OID" not in doc_dict or not doc_dict['stripped_OID']:
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} because it's missing an OID!!")
            skipped +=1
            continue


        stt_file = f"{doc_dict['stripped_OID']}_STT.xml"
        if stt_file not in all_alias_stt:
            stt_text_file = f"{doc_dict['stripped_OID']}_STT.txt"
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} with OID {doc_dict['OID']} because it's missing its XML file!! (stt exists: {stt_text_file in all_alias_stt})")
            skipped +=1
            continue
        else:
            doc_dict['stt_filename'] = stt_file

        # Extract PARTICIPANTS (list of dicts)
        participants = []
        for participant in doc_elem.find_all('PARTICIPANT'):
            name_elem = participant.find('NAME')
            function_elem = participant.find('FUNCTION')
            role_elem = participant.find('ROLE')
            participants.append({
                'name': name_elem.get_text(strip=True) if name_elem else None,
                'function': function_elem.get_text(strip=True) if function_elem else None,
                'role': role_elem.get_text(strip=True) if role_elem else None
            })
        if participants:
            doc_dict['participants'] = participants
        
        # Extract list fields (DESCRIPTORS, PROGRAMMES, SUBDOMAINS, etc.)
        list_fields = {
            'GEOGRAPHICALDESCRIPTORS': 'GEOGRAPHICALDESCRIPTOR',
            'PERSONDESCRIPTORS': 'PERSONDESCRIPTOR',
            'THEMATICALDESCRIPTORS': 'THEMATICALDESCRIPTOR',
            'RIGHTSUSAGEPOSSIBILITIES': 'RIGHTSUSAGEPOSSIBILITY',
            'PROGRAMMES': 'PROGRAMME',
            'SUBDOMAINS': 'SUBDOMAIN',
            'RECORDINGDATES': 'RECORDINGDATE',
            'FIRSTBROADCASTDATES': 'FIRSTBROADCASTDATE'
        }
        
        for container_name, renamed_field in list_fields_map.items():
            container = doc_elem.find(container_name)
            if container:
                doc_dict[renamed_field] = [elem.get_text(strip=True) for elem in container.find_all(list_fields[container_name])]
            else:
                # always define fields, set them to None if not defined
                doc_dict[renamed_field] = None

        date_strings = None
        broadcast_date = None
        # extract the date from first_braodcast_dates
        if doc_dict["first_broadcast_dates"]:
            date_strings = [d for rd in doc_dict["first_broadcast_dates"] for d in rd.split(" - ") if d != "__/__/____"]
            broadcast_date = True 
            
        if not date_strings and doc_dict["recording_dates"]:
            print(f"Did not find any date for doc with OID {doc_dict['OID']}. Trying to use the recording date. doc_dict: {doc_dict}")
            date_strings = [d for rd in doc_dict["recording_dates"] for d in rd.split(" - ") if d != "__/__/____" and "Avant" not in d and "Apr?s" not in d]
            broadcast_date = False

        # process the dates extracted
        if not date_strings:
            print(f"WARNING! MISSING DATE FOR DOC WITH OID {doc_dict['OID']}!! \nDocument: {doc_dict}, \noriginal: {doc_elem}")
            if doc_dict['stripped_OID'] == "5D740FA1-1A90-4878-A7E2-40C43C0F6430":
                print(f"Found and setting manually the correct date for {doc_dict['OID']}: 10/04/1989")
                doc_dict['date_str'] = "10/04/1989"
                doc_dict['exact_date'] = True
                doc_dict['broadcast_date'] = True
            #doc_dict['day'] = None
        else:
            # reformat each date and keep the earliest
            dates = []
            exact_dates = []
            for s in date_strings:
                exact = True
                if "__" in s:
                    old_s = s
                    s = s.replace("__", "01")
                    exact = False
                    print(f"The date for document with OID {doc_dict['OID']} was invalid ({old_s}) - changed it to {s}")
                if s.startswith('~'):
                    final_d = datetime.strptime(s[1:],  "%d/%m/%Y")
                else:   
                    final_d = datetime.strptime(s,  "%d/%m/%Y")
                dates.append(final_d)
                if exact:
                    exact_dates.append(final_d)
                

                    
            #dates = [datetime.strptime(s[1:] if s.startswith('~') else s, "%d/%m/%Y") for s in date_strings]
            #doc_dict['year'] = min(dates).year
            #doc_dict['month'] = min(dates).month
            #doc_dict['day'] = min(dates).day
            chosen_date = min(dates)
            doc_dict['date_str'] = chosen_date.strftime('%d/%m/%Y')
            doc_dict['exact_date'] = chosen_date in exact_dates
            doc_dict['broadcast_date'] = broadcast_date
            
        
        # Extract SUPPORTS (audio files and metadata)
        supports = []
        mp3_filenames = []

        for support in doc_elem.find_all('SUPPORT'):
            support_dict = {
                'spt_clsid': support.find('CLSID').get_text(strip=True) if support.find('CLSID') else None,
                'spt_oid': support.find('OID').get_text(strip=True) if support.find('OID') else None,
                'spt_title': support.find('TITLE').get_text(strip=True) if support.find('TITLE') else None,
                'spt_isdigital': support.find('ISDIGITAL').get_text(strip=True) if support.find('ISDIGITAL') else None,
                'spt_filename': support.find('FILENAME').get_text(strip=True) if support.find('FILENAME') else None,
                'spt_cataloguing_status': support.find('CATALOGUINGSTATUS').get_text(strip=True) if support.find('CATALOGUINGSTATUS') else None,
                'spt_source': support.find('SOURCE').get_text(strip=True) if support.find('SOURCE') else None,
                'spt_unit_duration': support.find('UNITDURATION').get_text(strip=True) if support.find('UNITDURATION') else None
            }

            if support_dict['spt_filename']:
                # remove '.wav' if it's in the filename
                spt_filename = support_dict['spt_filename'].replace('.wav', "")
                # populate the list of mp3 filenames with any mp3 file which has a matching filename
                mp3_filenames.extend([audio for audio in all_alias_audios if spt_filename in audio])

            supports.append(support_dict)

        if not supports or not mp3_filenames:
            print(f"Skipping document {doc_idx+1}/{len(doc_elements)} with OID {doc_dict['OID']} because it's missing its audio MP3 file!!")
            skipped +=1
            continue

        doc_dict['supports'] = supports
        doc_dict['spt_filenames'] = [s['spt_filename'] for s in supports]
        doc_dict['mp3_filenames'] = mp3_filenames

        # before adding to the list of docs, check it has all keys, and setting any missing one to None
        for k in doc_keys:
            if k not in doc_dict:
                doc_dict[k] = None
        
        documents.append(doc_dict)

    print(f"{program} - returning {len(documents)} documents, {skipped} were skipped due to missing necessary info.")
    
    return documents

Check that the function works correctly

In [9]:

audio_files = os.listdir(os.path.join(example_program_dir, audios_subdir))
text_files = os.listdir(os.path.join(example_program_dir, asr_subdir))

# Parse all documents from the example XML
all_documents = parse_docs_in_xml(xml_doc, example_program, text_files, audio_files)

print(f"Total documents found: {len(all_documents)}")
if all_documents:
    for idx, doc in enumerate(all_documents):
        #doc = all_documents[0]
        #print(f"\nFirst document keys: {list(doc.keys())}")
        print(f"\nTitle: {doc.get('title', 'N/A')[:80]}...")
        print(f"Broadcast program name: {doc.get('broadcast_program_name', 'N/A')}")
        print(f"Extracted boradcast date: {doc['date_str']} (year {doc['date_str'].split('/')[-1]})")
        print(f"Bradcast dates: {doc.get('first_broadcast_dates', ['N/A'])[0]}, Recording dates: {doc.get('recording_dates', ['N/A'])[0]}")
        print(f"MP3 filenames: {doc.get('mp3_filenames', ['N/A'])}")
        
        if doc['mp3_filenames']:
            for f in doc.get('mp3_filenames'):
                print(f" --> MP3 filename in audios: {f in audio_files}, OID in ASR files: {any(doc['stripped_OID'] in xml_f for xml_f in text_files)}")
        else:
            print(f"Doc {idx} has no mp3 filenames!! full doc:\n{doc}")
        #if doc.get('supports'):
        #    print(f"Audio file: {doc['supports'][0].get('filename', 'N/A')}")



Starting extracting 8 documents for program causerie_uni
Skipping document 3/8 with OID {1656BB3F-A8EF-4BF4-BE9A-20483F5665C8} because it's missing its XML file!! (stt exists: False)
Skipping document 5/8 with OID {87381313-87FD-4D31-AD19-8CDC08BA94DA} because it's missing its XML file!! (stt exists: False)
causerie_uni - returning 6 documents, 2 were skipped due to missing necessary info.
Total documents found: 6

Title: N/A...
Broadcast program name: Causerie universitaire
Extracted boradcast date: 19/12/1939 (year 1939)
Bradcast dates: 19/12/1939 - __/__/____, Recording dates: 20/11/1939 - 20/11/1939
MP3 filenames: ['3267f04d-657f-4dcb-bcb0-44b2b6c2682d_1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.mp3']
 --> MP3 filename in audios: True, OID in ASR files: True

Title: N/A...
Broadcast program name: Causerie universitaire
Extracted boradcast date: 07/01/1941 (year 1941)
Bradcast dates: 07/01/1941 - __/__/____, Recording dates: 26/11/1940 - 26/11/1940


In [10]:
all_documents[:5]

[{'alias': 'causerie_uni',
  'date_str': '19/12/1939',
  'mp3_filenames': ['3267f04d-657f-4dcb-bcb0-44b2b6c2682d_1211554131-1466X_complet_wav_958-SIROM{CFEC57B3-AADF-47BB-8ACF-49BE06EB6AD3}.mp3'],
  'cls_ID': '{D2593F4E-C887-4E48-8982-5BD08BA4DAE0}',
  'OID': '{3267F04D-657F-4DCB-BCB0-44B2B6C2682D}',
  'stripped_OID': '3267F04D-657F-4DCB-BCB0-44B2B6C2682D',
  'login': '',
  'broadcast_episode_title': "Adolphe Dygasinski, un Kipling avant Kipling. Causerie de Edouard Cros, privat-docent ? l'Universit? de Fribourg",
  'sequence': '1',
  'broadcast_program_name': 'Causerie universitaire',
  'document_type': 'Parl?',
  'hierarchy_level': 'Sujet',
  'modified_by': 'albrecjo',
  'modified_on': '25.05.2022 03:56:53',
  'physical_support_history': 'Disques 78T maison Radio-Lausanne 1466X A, 1467X A, 1466X B, 1467X B',
  'production_type': 'Production propre',
  'recording_place': 'Lausanne (Studio de Radio-Lausanne)',
  'rights_notes': 'Memoriav',
  'rights_status': 'Clarifi?',
  'series_title

In [171]:
doc_keys

dict_keys(['alias', 'date_str', 'mp3_filenames', 'cls_ID', 'OID', 'stripped_OID', 'login', 'broadcast_episode_title', 'sequence', 'broadcast_program_name', 'document_type', 'hierarchy_level', 'modified_by', 'modified_on', 'physical_support_history', 'production_type', 'recording_place', 'rights_notes', 'rights_status', 'series_title', 'content_summary', 'workflow_status', 'assembly_status', 'live', 'modilation_type', 'work_duration', 'work_duration_compl', 'participants', 'geographical_descriptors', 'person_descriptors', 'thematical_descriptors', 'rights_uage_possibilities', 'radio_channels', 'subdomains', 'recording_dates', 'first_broadcast_dates', 'supports', 'spt_filenames'])

In [11]:
doc_keys = all_documents[0].keys()

out_csv_name = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/debug_metadata.csv"
with open(out_csv_name, "w", newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, doc_keys)
    dict_writer.writeheader()
    dict_writer.writerows(all_documents)


This works and yields the desired set of metadata. 

## Aggregating all the programs' metadata into a single file

Now we need to write an orchestrator function that opens and processes the XML documents for each provider, and stores the info in a dict or a dataframe, also extracting the first braodcast date to help with the creation of the issue index file

In [7]:
all_program_aliases = sorted([p for p in os.listdir(base_dir) if "DS_Store" not in p and "$RECYCLE" not in p and "System" not in p])
print(f"Found {len(all_program_aliases)} Radio programs: {all_program_aliases}")

Found 47 Radio programs: ['ana_media', 'bigbang', 'canal_euro', 'causerie_uni', 'chron_instit', 'chron_unesco', 'courrier_cr', 'culte', 'dos_sci', 'ecoute_paix', 'enquest', 'forum', 'forum_lau', 'geneve_info', 'hist_ondes', 'infopile', 'inst_monde', 'j13h', 'j13h2', 'j_mat', 'j_midi', 'j_nuit', 'j_soir', 'mag_eco', 'mag_info', 'mag_sci1', 'mag_sci2', 'mag_tv1', 'mag_tv2', 'mem_ondes', 'min_oecu', 'miroir_monde', 'miroir_temps', 'monde_ant', 'monde_sem', 'nickel', 'nu_parle', 'ombres_eco', 'paraboles', 'paris_parle', 'parole_prem', 'petitdej', 'suisse_euro', 'terre_ciel', 'trib_prem', 'vie_monde', 'vie_va']


In [8]:
out_csv_name = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata.rts.csv"

In [184]:
all_program_aliases.index("mem_ondes")

29

In [29]:
already_done = all_program_aliases[:all_program_aliases.index("mem_ondes")+1]
already_done = []

In [27]:
#already_done = all_program_aliases[:all_program_aliases.index("mem_ondes")+1]
#already_done = []
already_done

[]

In [ ]:
all_docs = []
all_docs_dict = {}
#doc_keys = None

for p_idx, p_alias in tqdm(enumerate(all_program_aliases)):

    if p_alias in already_done:
        print(f"\n{p_alias} is already done, skipping!")
        continue
    
    # first find all the metadata xml docs
    program_dir = os.path.join(base_dir, p_alias)
    metadata_files = [os.path.join(program_dir,f) for f in os.listdir(program_dir) if f.startswith(metadata_file_start)]
    
    all_audios_for_alias = os.listdir(os.path.join(program_dir,'audio'))
    all_stt_for_alias = os.listdir(os.path.join(program_dir,'stt'))

    print(f"\nPROCESSING PROGRAM {p_alias} ({p_idx+1}/{len(all_program_aliases)}) - {len(metadata_files)} files:")

    program_docs = []
    for xml_doc_path in tqdm(metadata_files):
        with open(xml_doc_path, "r", encoding="utf-8") as f:
            raw_xml = f.read()

        program_docs.extend(parse_docs_in_xml(BeautifulSoup(raw_xml, "xml"), p_alias, all_stt_for_alias, all_audios_for_alias, doc_keys))
    
    all_docs.extend(program_docs)
    all_docs_dict[p_alias] = program_docs

    # Save the current list of documents to save the progress
    #if not doc_keys:
    #    doc_keys = all_docs[0].keys()

    print(f" --> Adding {len(program_docs)} to the out csv for {p_alias}")
    with open(out_csv_name, "a", newline='') as output_file:
        dict_writer = csv.DictWriter(output_file, doc_keys)
        if not already_done:
            dict_writer.writeheader()
        dict_writer.writerows(program_docs)

    already_done.append(p_alias)
        


There were many duplicates due to a small mistake: the rows were all written again for each program

In [18]:
metadata_df = pd.read_csv(out_csv_name)
#metadata_df = metadata_df.drop_duplicates()
len(metadata_df)

26197

In [17]:
metadata_df.to_csv(out_csv_name, index=True)

In [10]:
v1_csv_name = out_csv_name.replace("rts", "rts_v1")
v1_csv_name

'/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata.rts_v1.csv'

In [20]:
metadata_v1_df = pd.read_csv(v1_csv_name)
metadata_v1_df = metadata_v1_df.drop_duplicates()
len(metadata_v1_df)

36856

### Claude fix of the corrupted text where '?' replaces characters with accents

In [ ]:

# Now I understand all failures. Most of these are in MANUAL_MAP already.
# The rule failures are for tokens NOT in the manual map.
# Let me add the remaining edge cases to MANUAL_MAP and tighten the rules:
#
# Issues:
# 1. "St?r?o" - MANUAL_MAP has it, works ✓
# 2. "?lections" - MANUAL_MAP has it, works ✓ (same for all ?-initial words)
# 3. "Conf?rence" - MANUAL_MAP has it ✓
# 4. "m?me" - MANUAL_MAP has it ✓
# 5. "si?cle" - MANUAL_MAP has it ✓
# 6. "for?ts" - MANUAL_MAP has it ✓
# 7. "fa?on" - MANUAL_MAP has it ✓
# 8. "re?u" - MANUAL_MAP has it ✓
#
# ALL failures in the test are for tokens that ARE in the manual map!
# The test was using correct_by_rules directly without checking manual map first.
# The actual correct_token function checks manual map first, then falls back to rules.
# So the test was misleading.
#
# The ONLY critical rule fix is:
# 1. ç rule: '' in 'aou' = True -> need "after != '' and after.lower() in 'aou'"
# 2. à rule: fires before word-initial check - need to only fire when it's truly standalone
#    (not at position 0 of a word-token, since word-tokens don't start with spaces)
#    Actually, word-tokens extracted by re.split never have spaces, so before='' means i=0
#    Fix: only apply à rule if before=' ' (in context, not token-start)
#    Since we extract pure word tokens, before='' always means start of token -> should be é/è
#
# Let me write the final clean version and apply it.

import pandas as pd, re

# ─── FINAL MANUAL MAP (comprehensive) ───
MANUAL_MAP = {
    "?": "à",
    "Parl?": "Parlé", "'Parl?": "'Parlé", "Clarifi?": "Clarifié", "Valid?": "Validé",
    "Premi?re'": "Première'", "Accept?": "Accepté", "Mont?": "Monté",
    "Premi?re": "Première", "St?r?o": "Stéréo",
    "'Pr?sentateur": "'Présentateur", "Pr?sentateur": "Présentateur",
    "'Interview?": "'Interviewé", "Interview?": "Interviewé",
    "Gen?ve": "Genève", "'Gen?ve": "'Genève", "Gen?ve'": "Genève'",
    "Radio-Gen?ve": "Radio-Genève", "Radio-Gen?ve'": "Radio-Genève'",
    "Neuch?tel": "Neuchâtel", "'Neuch?tel": "'Neuchâtel", "Neuch?tel'": "Neuchâtel'",
    "neuch?telois": "neuchâtelois", "neuch?teloise": "neuchâteloise",
    "B?le": "Bâle", "Del?mont": "Delémont", "Z?rich": "Zürich",
    "Montr?al": "Montréal", "J?rusalem": "Jérusalem", "J?rusalem'": "Jérusalem'",
    "T?h?ran": "Téhéran", "Qu?bec": "Québec",
    "Alg?rie": "Algérie", "'Alg?rie'": "'Algérie'", "d'Alg?rie": "d'Algérie",
    "l'Alg?rie": "l'Algérie", "P?kin": "Pékin", "Cor?e": "Corée",
    "Ha?ti": "Haïti", "Za?re": "Zaïre", "Isra?l": "Israël",
    "'Isra?l'": "'Israël'", "d'Isra?l": "d'Israël", "Gr?ce": "Grèce",
    "Su?de": "Suède", "Kowe?t": "Koweït", "Am?rique": "Amérique",
    "Bosnie-Herz?govine": "Bosnie-Herzégovine",
    "'Bosnie-Herz?govine'": "'Bosnie-Herzégovine'", "Herz?govine": "Herzégovine",
    "Andr?": "André", "Andr?'": "André'", "d'Andr?": "d'André",
    "Ren?": "René", "Ren?'": "René'", "Jean-Ren?'": "Jean-René'",
    "Fran?ois": "François", "Fran?ois'": "François'",
    "Jean-Fran?ois": "Jean-François", "Jean-Fran?ois'": "Jean-François'",
    "Fran?oise": "Françoise", "Fran?oise'": "Françoise'",
    "G?rard": "Gérard", "G?rard'": "Gérard'",
    "Rapha?l": "Raphaël", "Rapha?l'": "Raphaël'",
    "Jos?": "José", "Mich?le": "Michèle", "Mich?le'": "Michèle'",
    "Marie-Jos?'": "Marie-José'", "L?on": "Léon", "L?o": "Léo",
    "Fr?d?ric": "Frédéric", "Fr?d?ric'": "Frédéric'",
    "G?rald": "Gérald", "F?lix": "Félix", "F?licien": "Félicien",
    "Mikha?l": "Mikhaïl", "J?rg": "Jörg", "J?rg'": "Jörg'",
    "Val?ry": "Valéry", "R?my": "Rémy", "G?n?ral": "Général",
    "No?l": "Noël", "Genevi?ve": "Geneviève", "B?at'": "Béat'",
    "C?cile'": "Cécile'", "Marl?ne'": "Marlène'",
    "Pierre-Andr?": "Pierre-André", "Pierre-Andr?'": "Pierre-André'",
    "Georges-Andr?": "Georges-André", "Georges-Andr?'": "Georges-André'",
    "St?phane": "Stéphane", "St?phane'": "Stéphane'",
    "Fran?ais": "Français", "Ma?tre": "Maître",
    "B?guin": "Béguin", "'B?guin": "'Béguin", "B?guelin": "Béguelin", "'B?guelin": "'Béguelin",
    "Fran?ois-Achille'": "François-Achille'",
    "Nussl?": "Nusslé", "'Nussl?": "'Nusslé", "Werl?": "Werlé", "'Werl?": "'Werlé",
    "K?nzler": "Künzler", "'K?nzler": "'Künzler",
    "M?ller": "Müller", "'M?ller": "'Müller",
    "Gilli?ron": "Gilliéron", "'Gilli?ron": "'Gilliéron",
    "Moss?": "Mossé", "'Moss?": "'Mossé",
    "D?caillet": "Décaillet", "'D?caillet": "'Décaillet",
    "D?cotte": "Décotte", "'D?cotte": "'Décotte",
    "pr?sident": "président", "'pr?sident": "'président",
    "vice-pr?sident": "vice-président", "'vice-pr?sident": "'vice-président",
    "Pr?sident": "Président", "Pr?sentation": "Présentation",
    "pr?sidente": "présidente", "pr?sidentielle": "présidentielle",
    "pr?sidentielle'": "présidentielle'", "pr?sidentielles": "présidentielles",
    "pr?sidence": "présidence", "pr?sent": "présent",
    "pr?sent?": "présenté", "pr?sente": "présente", "pr?sentation": "présentation",
    "f?d?ral": "fédéral", "f?d?ral'": "fédéral'", "f?d?rale": "fédérale",
    "f?d?rale'": "fédérale'", "f?d?rales": "fédérales",
    "f?d?ration": "fédération", "F?d?ration": "Fédération", "F?d?ral": "Fédéral",
    "Conf?d?ration": "Confédération", "Conf?d?ration'": "Confédération'",
    "fran?ais": "français", "fran?ais'": "français'",
    "fran?aise": "française", "fran?aise'": "française'", "fran?aises": "françaises",
    "d?claration": "déclaration", "D?claration": "Déclaration",
    "'D?claration": "'Déclaration", "'D?claration'": "'Déclaration'",
    "d?clarations": "déclarations",
    "apr?s": "après", "Apr?s": "Après", "Apr?s'": "Après'", "aupr?s": "auprès",
    "g?n?ral": "général", "g?n?rale": "générale",
    "?t?": "été", "?crivain": "écrivain", "'?crivain'": "'écrivain'",
    "'?crivain": "'écrivain", "?crivain'": "écrivain'", "l'?crivain": "l'écrivain",
    "?crivaine": "écrivaine", "'?crivaine'": "'écrivaine'", "?crivaine'": "écrivaine'",
    "secr?taire": "secrétaire", "'secr?taire": "'secrétaire",
    "D?partement": "Département", "d?partement": "département",
    "?me": "ème", "Conf?rence": "Conférence", "'Conf?rence": "'Conférence",
    "conf?rence": "conférence", "sp?cial": "spécial", "sp?ciale": "spéciale",
    "sp?cialiste": "spécialiste", "l'Universit?": "l'Université",
    "l'universit?": "l'université",
    "?conomique": "économique", "?conomique'": "économique'",
    "?conomiques": "économiques", "?conomie": "économie",
    "l'?conomie": "l'économie", "'?conomie'": "'économie'",
    "d'?conomie": "d'économie", "?conomiste": "économiste", "?conomiste'": "économiste'",
    "R?action": "Réaction", "r?action": "réaction", "r?actions": "réactions",
    "R?actions": "Réactions", "R?publique": "République", "r?publique": "république",
    "T?moignage": "Témoignage", "t?moignage": "témoignage",
    "t?moignages": "témoignages", "T?moignages": "Témoignages", "t?moin": "témoin",
    "?tre": "être", "d'?tre": "d'être", "Pr?t": "Prêt", "pr?t": "prêt",
    "pr?ts": "prêts", "pr?tre": "prêtre", "'pr?tre'": "'prêtre'", "pr?tres": "prêtres",
    "?missions": "émissions", "l'?mission": "l'émission", "'?mission": "'émission",
    "?mission": "émission",
    "premi?re": "première", "premi?res": "premières",
    "oecum?nique": "œcuménique", "'oecum?nisme'": "'œcuménisme'",
    "sovi?tique": "soviétique", "sovi?tiques": "soviétiques", "Sovi?tiques": "Soviétiques",
    "am?ricain": "américain", "am?ricaine": "américaine",
    "am?ricains": "américains", "am?ricaines": "américaines", "Am?ricains": "Américains",
    "l'arm?e": "l'armée", "'arm?e'": "'armée'", "arm?e": "armée", "arm?'": "armé'",
    "soci?t?": "société", "Soci?t?": "Société", "soci?t?s": "sociétés",
    "'Deuxi?me": "'Deuxième", "Deuxi?me": "Deuxième", "deuxi?me": "deuxième",
    "r?fugi?s": "réfugiés", "r?fugi?": "réfugié", "r?fugi?'": "réfugié'",
    "'r?fugi?'": "'réfugié'", "r?fugi?e": "réfugiée",
    "?lections": "élections", "'?lection": "'élection", "?lection": "élection",
    "'?lection'": "'élection'", "l'?lection": "l'élection", "?lectorale": "électorale",
    "isra?lien": "israélien", "isra?lienne": "israélienne",
    "r?gion": "région", "r?gions": "régions", "n?gociations": "négociations",
    "c?r?monie": "cérémonie", "C?r?monie": "Cérémonie",
    "m?me": "même", "m?mes": "mêmes", "sc?ne": "scène",
    "Comit?": "Comité", "comit?": "comité",
    "g?n?rale": "générale", "com?dien": "comédien",
    "'com?dien'": "'comédien'", "'com?dien": "'comédien",
    "com?dienne": "comédienne", "'com?dienne'": "'comédienne'",
    "Congr?s": "Congrès", "congr?s": "congrès",
    "l'ann?e": "l'année", "ann?e": "année", "ann?es": "années",
    "Enqu?te": "Enquête", "d'enqu?te": "d'enquête", "enqu?te": "enquête",
    "l'enqu?te": "l'enquête", "d?part": "départ", "march?": "marché",
    "March?": "Marché", "d?mission": "démission", "D?mission": "Démission",
    "d?but": "début", "D?but": "Début", "charg?": "chargé", "d?fense": "défense",
    "gr?ve": "grève", "'gr?ve'": "'grève'",
    "l'Assembl?e": "l'Assemblée", "l'assembl?e": "l'assemblée",
    "communaut?": "communauté", "Communaut?": "Communauté", "r?gime": "régime",
    "?lu": "élu", "?lus": "élus", "?lue": "élue",
    "d?veloppement": "développement", "po?te": "poète",
    "autorit?s": "autorités", "cons?quences": "conséquences", "libert?": "liberté",
    "th??tre": "théâtre", "Th??tre": "Théâtre",
    "'cin?ma'": "'cinéma'", "cin?ma": "cinéma",
    "m?re": "mère", "P?re": "Père", "p?re": "père",
    "fr?re": "frère", "fr?res": "frères",
    "F?te": "Fête", "f?te": "fête", "'f?te": "'fête", "f?tes": "fêtes",
    "f?vrier": "février",
    "?tat": "État", "d'?tat": "d'État", "l'?tat": "l'État", "?tats": "États",
    "t?te": "tête", "bless?s": "blessés", "r?alisateur": "réalisateur",
    "d?bat": "débat", "D?bat": "Débat", "'D?bat'": "'Débat'",
    "d?bat'": "débat'", "D?bat'": "Débat'",
    "ch?mage": "chômage", "'ch?mage'": "'chômage'", "ch?meurs": "chômeurs",
    "pr?sence": "présence", "pass?": "passé", "journ?e": "journée", "Journ?e": "Journée",
    "l'?tranger": "l'étranger", "ing?nieur": "ingénieur", "d?s": "dès",
    "?ge": "âge", "l'?ge": "l'âge",
    "?galement": "également", "?v?nements": "événements", "'?v?nement": "'événement",
    "difficult?s": "difficultés", "D?c?s": "Décès", "d?c?s": "décès",
    "d?cembre": "décembre",
    "l'?cole": "l'école", "?cole": "école", "'?coles": "'écoles", "?coles": "écoles",
    "re?u": "reçu", "contr?le": "contrôle",
    "r?union": "réunion", "R?union": "Réunion", "'r?union": "'réunion",
    "solidarit?": "solidarité", "Solidarit?": "Solidarité",
    "si?ge": "siège", "l'a?roport": "l'aéroport",
    "a?rienne": "aérienne", "a?rienne'": "aérienne'", "a?rien'": "aérien'",
    "a?rostier": "aérostier", "ao?t": "août",
    "cha?ne": "chaîne", "Cha?ne": "Chaîne",
    "condamn?": "condamné", "r?sistance": "résistance",
    "?tudiants": "étudiants", "?tudiant": "étudiant",
    "r?forme": "réforme", "r?formes": "réformes", "d?mocratie": "démocratie",
    "'d?mocratie": "'démocratie", "carri?re": "carrière",
    "ext?rieur": "extérieur", "ext?rieure": "extérieure",
    "l?gislatives": "législatives", "'l?gislation'": "'législation'",
    "'l?gislatif": "'législatif",
    "?v?que": "évêque", "?v?ques": "évêques",
    "?trangers": "étrangers", "?trang?res": "étrangères",
    "?trang?re": "étrangère", "?trang?res'": "étrangères'",
    "fronti?res": "frontières", "fronti?re": "frontière",
    "r?f?rendum": "référendum", "'r?f?rendum'": "'référendum'",
    "?voque": "évoque", "'ex?cutif": "'exécutif",
    "Mus?e": "Musée", "mus?e": "musée", "pi?ce": "pièce",
    "Pr?dication": "Prédication", "pr?dication": "prédication",
    "r?volution": "révolution", "litt?rature": "littérature",
    "'litt?rature'": "'littérature'", "litt?raire": "littéraire",
    "litt?raire'": "littéraire'",
    "l'ind?pendance": "l'indépendance", "'ind?pendantisme'": "'indépendantisme'",
    "r?daction": "rédaction", "mati?re": "matière", "donn?": "donné",
    "d?cid?": "décidé", "syst?me": "système", "succ?s": "succès",
    "r?vision": "révision", "d?couverte": "découverte",
    "arr?t?": "arrêté", "Arr?t": "Arrêt",
    "financi?re": "financière", "sign?": "signé",
    "m?decine": "médecine", "'m?decine'": "'médecine'",
    "m?decin": "médecin", "'m?decin": "'médecin", "'m?decin'": "'médecin'",
    "m?decins": "médecins", "M?decins": "Médecins",
    "T?l?vision": "Télévision", "'t?l?vision'": "'télévision'",
    "t?l?vision": "télévision", "t?l?phone": "téléphone",
    "derni?re": "dernière", "derni?res": "dernières",
    "l'int?rieur": "l'intérieur", "l'Int?rieur": "l'Intérieur",
    "l'arriv?e": "l'arrivée", "arriv?e": "arrivée", "Arriv?e": "Arrivée",
    "?taient": "étaient", "?tait": "était",
    "M?moire": "Mémoire", "m?moire": "mémoire",
    "Lib?ration": "Libération", "lib?ration": "libération",
    "lib?ral": "libéral", "lib?rale": "libérale",
    "cr?ation": "création", "Cr?ation": "Création",
    "envoy?": "envoyé", "'envoy?": "'envoyé",
    "d?l?gation": "délégation", "d?l?gu?": "délégué",
    "'d?l?gu?": "'délégué", "d?l?gu?s": "délégués", "'d?l?gu?s": "'délégués",
    "r?le": "rôle", "r?les": "rôles",
    "s?curit?": "sécurité", "proc?s": "procès", "'proc?s'": "'procès'",
    "r?dacteur": "rédacteur", "'r?dacteur": "'rédacteur",
    "si?cle": "siècle", "si?cle'": "siècle'",
    "d?put?": "député", "'d?put?": "'député", "d?put?e": "députée",
    "d?put?s": "députés", "d?put?es": "députées",
    "d?cision": "décision", "D?cision": "Décision",
    "l'?nergie": "l'énergie", "'?nergie": "'énergie", "?nergies": "énergies",
    "repr?sentant": "représentant", "repr?sentants": "représentants",
    "conseill?re": "conseillère", "'conseill?re": "'conseillère",
    "d'?tat": "d'État", "pr?s": "près",
    "comp?tition": "compétition", "sant?": "santé",
    "R?trospective": "Rétrospective", "c?t?": "côté", "m?dias": "médias",
    "l'adh?sion": "l'adhésion", "r?sultats": "résultats", "r?sultat": "résultat",
    "Europ?enne": "Européenne", "europ?enne": "européenne",
    "europ?enne'": "européenne'", "europ?ennes": "européennes",
    "europ?en": "européen", "europ?ens": "européens",
    "malgr?": "malgré", "coop?ration": "coopération",
    "th?ologie": "théologie", "th?ologien": "théologien",
    "alg?rien": "algérien", "minist?re": "ministère",
    "d?": "dé", "d?mocrate-chr?tien": "démocrate-chrétien",
    "pr?nom": "prénom", "chr?tiens": "chrétiens", "chr?tien": "chrétien",
    "th?me": "thème", "employ?s": "employés",
    "f?d?ration": "fédération", "organis?": "organisé",
    "S?rie": "Série", "s?rie": "série",
    "diff?rents": "différents", "diff?rentes": "différentes",
    "diff?rent": "différent", "m?tier": "métier",
    "entra?neur": "entraîneur", "laur?at": "lauréat",
    "consacr?": "consacré", "consacr?e": "consacrée",
    "?tudes": "études", "d'?tudes": "d'études", "?tude": "étude",
    "?pouse": "épouse", "r?ponse": "réponse",
    "n?cessit?": "nécessité", "?meutes": "émeutes",
    "l'?le": "l'île", "for?ts": "forêts",
    "m?dicale": "médicale", "responsabilit?": "responsabilité",
    "personnalit?s": "personnalités", "personnalit?": "personnalité",
    "op?ration": "opération", "Op?ration": "Opération", "op?rations": "opérations",
    "proc?dure": "procédure", "d?tenus": "détenus", "trouv?": "trouvé",
    "pr?vention": "prévention", "publicit?": "publicité",
    "neutralit?": "neutralité", "r?unification": "réunification",
    "nucl?aire": "nucléaire", "nucl?aire'": "nucléaire'", "nucl?aires": "nucléaires",
    "l'?quipe": "l'équipe", "?change": "échange", "o?": "où", "tr?s": "très",
    "probl?me": "problème", "probl?mes": "problèmes",
    "nomm?": "nommé", "strat?gie": "stratégie",
    "activit?": "activité", "activit?s": "activités",
    "exp?rience": "expérience", "propri?taire": "propriétaire",
    "Nestl?": "Nestlé", "annonc?": "annoncé", "priv?e": "privée",
    "col?re": "colère", "accept?": "accepté",
    "Tch?coslovaquie": "Tchécoslovaquie", "rencontr?": "rencontré",
    "Di?te": "Diète", "lanc?": "lancé", "cr?er": "créer",
    "Proc?s": "Procès", "m?daille": "médaille",
    "p?dophile": "pédophile", "horlog?re": "horlogère",
    "fa?on": "façon", "reconna?t": "reconnaît",
    "al?manique": "alémanique", "v?cu": "vécu",
    "'comm?moration'": "'commémoration'", "p?trole": "pétrole",
    "lumi?res": "lumières", "l'abb?": "l'abbé", "l'h?pital": "l'hôpital",
    "mani?re": "manière", "helv?tique": "helvétique",
    "d?mocrate": "démocrate", "?l?ments": "éléments",
    "d?nonce": "dénonce", "d?faite": "défaite", "plut?t": "plutôt",
    "gr?ce": "grâce", "volont?": "volonté", "b?timent": "bâtiment",
    "l'op?ration": "l'opération", "stup?fiants'": "stupéfiants'",
    "jusqu'?": "jusqu'à",
    "requ?rant": "requérant", "requ?rants": "requérants",
    "assassin?": "assassiné", "r?alit?": "réalité",
    "l'Acad?mie": "l'Académie", "d?cide": "décide",
    "cin?aste": "cinéaste", "d?tention": "détention",
    "l'?volution": "l'évolution", "l'?gard": "l'égard",
    "?glise": "église", "l'?glise": "l'église", "?glises": "églises",
    "'?glise": "'église",
    "R?cit": "Récit", "r?cit": "récit", "'R?citant": "'Récitant",
    "?crit": "écrit", "?crits": "écrits", "?crite": "écrite",
    "f?minin": "féminin", "occup?s": "occupés",
    "majorit?": "majorité", "?tabli": "établi", "?tablie": "établie",
    "?le": "île", "?le'": "île'",
    "?lecteur": "électeur", "?lecteurs": "électeurs",
    "?lection": "élection", "?lection'": "élection'",
    "?lectricit?": "électricité",
    "?levage": "élevage", "?leveur": "éleveur", "?leveurs": "éleveurs",
    "?lite": "élite", "?loge": "éloge",
    "?boueur": "éboueur", "?branlement": "ébranlement",
    "?cart": "écart", "?chafaud": "échafaud",
    "?chec": "échec", "?checs'": "échecs'",
    "?chographie'": "échographie'", "?clairage": "éclairage",
    "?clipse'": "éclipse'", "?coli?re": "écolière",
    "?colier": "écolier", "?colier'": "écolier'", "?coliers": "écoliers",
    "?cologie": "écologie", "?cologie'": "écologie'", "?cologisme'": "écologisme'",
    "?conomie": "économie", "?conomie'": "économie'", "?conomies": "économies",
    "?coute": "écoute", "?craser": "écraser", "?crevisse'": "écrevisse'",
    "?criture": "écriture", "?criture'": "écriture'",
    "?crivain": "écrivain", "?crivains": "écrivains",
    "?dition": "édition", "?dition'": "édition'",
    "?ducateur": "éducateur", "?ducateur'": "éducateur'",
    "?ducateurs": "éducateurs", "?ducation": "éducation", "?ducation'": "éducation'",
    "?ducatrice": "éducatrice",
    "?galit?": "égalité", "?galit?'": "égalité'",
    "?gyptologue": "égyptologue", "?gyptologue'": "égyptologue'",
    "?largir": "élargir",
    "?l?ve'": "élève'", "?l?ves": "élèves", "?l?phant'": "éléphant'",
    "?cologiste": "écologiste", "?cologiste'": "écologiste'",
    "?ll?s": "elles",
    "d?j?": "déjà",
    "carr?": "carré",
    "t?l?vis?": "télévisé",
    "f?d?r?": "fédéré",
    "diff?renci?": "différencié",
    "?v?nement": "événement", "?v?nements": "événements",
    "?v?que": "évêque",
    "soci?t?": "société",
    "t?moignage": "témoignage",
    "repr?senter": "représenter",
    "s?curit?s": "sécurités",
    "r?sistance": "résistance",
    "th?orie": "théorie",
    "cin?matographique": "cinématographique",
    "po?sie": "poésie",
    "'po?sie'": "'poésie'",
    "int?gration": "intégration",
    "'int?gration": "'intégration",
    "l'int?gration": "l'intégration",
    "d?l?gu?s": "délégués",
    "'d?put?": "'député",
    "d?bats": "débats",
    "repr?sente": "représente",
    "abb?": "abbé",
    "t?l?": "télé",
    "?crivains": "écrivains",
    "?dition": "édition",
    "p?re": "père",
    "carr?": "carré",
    "arm?": "armé",
    "g?ographe": "géographe",
    "g?ographie": "géographie",
    "g?ographique": "géographique",
    "arch?ologue": "archéologue",
    "arch?ologie": "archéologie",
    "Universit?s": "Universités",
    "universit?s": "universités",
    "M?t?o": "Météo",
    "li?": "lié",
    "L'?volution": "L'évolution",
    "l'?ch?ance": "l'échéance",
    "adh?sion": "adhésion",
    "El?ment": "Elément",
    "el?ments": "eléments",
    "install?s": "installés",
    "chass?s": "chassés",
    "marqu?s": "marqués",
    "marqu?": "marqué",
    "cha?nes": "chaînes",
    "l'entra?neur": "l'entraîneur",
    "au-del?": "au-delà",
    "Jusqu'o?": "Jusqu'où",
    "men?s": "menés",
    "Plan?te": "Planète",
    "concr?tement": "concrètement",
    "d?veloppment": "développement",
    "J?r?me": "Jérôme",
    "requ?tes": "requêtes",
    # all the words ending in -s to remove the rule which broke many participe passés
    "proc?s": "procès",
    "apr?s": "après",
    "tr?s": "très",
    "succ?s": "succès",
    "exc?s": "excès",
    "acc?s": "accès",
    "progr?s": "progrès",
    "congr?s": "congrès",
    "d?c?s": "décès",
    "expr?s": "exprès",
    "aupr?s": "auprès",
    "pr?s": "près",
    "d?s": "dès",
    "th??tre": "théâtre", "Th??tre":"Théâtre"
}

# Suffixes after which a word-final ? is a real question mark, not an accent
_REAL_QUESTION_SUFFIXES = (
    # verb infinitives
    'er', 'ir', 're', 'dre', 'tre', 'ndre', 'indre', 'oudre',
    # common noun/adj endings that don't take accents
    'eur', 'eur', 'oir', 'our', 'eur', "istes",
    # explicit vowel endings
    'a', 'e', 'i', 'o', 'u', 'y',
)


def _is_real_question_mark(token, pos):
    """Return True if the ? at position pos in token is a genuine question mark."""
    # Only applies to word-final ?
    if pos != len(token) - 1:
        return False
    before = token[:pos].lower()
    # After a vowel directly
    if before and before[-1] in 'aeiouy':
        return True
    # After known suffixes that never take a final accent
    for suffix in _REAL_QUESTION_SUFFIXES:
        if before.endswith(suffix):
            return True
    return False

def correct_by_rules(token, char='?'):
    if char not in token:
        return token
    result = list(token)
    i = 0
    while i < len(result):
        if result[i] != char:
            i += 1
            continue
        before = result[i-1] if i > 0 else ''
        after = result[i+1] if i < len(result)-1 else ''
        after2 = result[i+2] if i < len(result)-2 else ''

        # ── Real question mark? (word-final after verb/noun endings) ──
        if char=='?' and _is_real_question_mark(token, i):
            result[i] = '?'

        # ç: ? before a/o/u after consonant (after MUST be non-empty)
        elif after and after.lower() in 'aou' and before.lower() in 'cfglmnst':
            # Restrict ç: mainly after n, c, l, s, g, f (not r which is rare)
            result[i] = 'ç'
        # è before -re at non-word-final position (ère endings)
        # Only if followed by 're' and that's near end of word
        elif after.lower() == 'r' and after2.lower() == 'e' and (i+3 >= len(result) or result[i+3] in ('s', '', 'z')):
            result[i] = 'è'
        # è before -me (problème, même) - ê for même is in manual map
        elif after.lower() == 'm' and after2.lower() == 'e':
            result[i] = 'è'
        # è before -ve (grève) - in manual map
        elif after.lower() == 'v' and after2.lower() == 'e':
            result[i] = 'è'
        # è before -ge (siège) - in manual map
        elif after.lower() == 'g' and after2.lower() == 'e':
            result[i] = 'è'
        # è before final -s (procès, après, très)
        #elif after.lower() == 's' and (i+2 >= len(result) or result[i+2] in (' ', '', "'", ',', '.', ')', ']', '"', '-')):
        #    result[i] = 'è'
        # è before -ce (pièce)
        elif after.lower() == 'c' and after2.lower() == 'e':
            result[i] = 'è'
        # è: i?r pattern (première)
        elif before.lower() == 'i' and after.lower() == 'r':
            result[i] = 'è'
        # è: ?v (Genève: n?v → è)
        elif after.lower() == 'v':
            result[i] = 'è'
        # ê before final -t (arrêt, forêt) - in manual map
        elif after.lower() == 't' and (i+2 >= len(result) or result[i+2] in (' ', '', "'", ',', '.', ')')):
            result[i] = 'ê'
        # ô after h
        elif before.lower() == 'h':
            result[i] = 'ô'
        # Word-initial (i=0 in token): ? → mostly é
        elif i == 0:
            if after.lower() == 'l' and after2 and after2.lower() not in 'aeioué':
                result[i] = 'î'   # île
            elif after.lower() == 'g' and after2.lower() == 'e':
                result[i] = 'â'   # âge
            else:
                result[i] = 'é'
        # Word-final ? after vowel 'e' → likely real question mark
        elif not after and before.lower() in 'aeiouy':
            result[i] = '?'   # Keep as real ?
        # Default: é
        else:
            result[i] = 'é'
        i += 1
    return ''.join(result)

def correct_token(token, char="?", _map = MANUAL_MAP):
    if token in _map:
        return _map[token]
    return correct_by_rules(token, char=char)

def fix_cell(cell_value, char="?", _map=MANUAL_MAP):
    if not isinstance(cell_value, str) or char not in cell_value:
        return cell_value
    if char == "?":
        parts = re.split(r"([A-Za-z?àâäéèêëîïôùûüçœæ'-]+)", cell_value)
    elif char == "�":
        parts = re.split(r"([A-Za-z�àâäéèêëîïôùûüçœæ'-]+)", cell_value)
    result = []
    for part in parts:
        if char in part and re.search(r'[A-Za-z]', part):
            result.append(correct_token(part, char=char, _map=_map))
        elif part == char:
            result.append('à')   # standalone ? is always "à"
        else:
            result.append(part)
    return ''.join(result)

ID_COLS = {
    'alias', 'date_str', 'stt_filename', 'mp3_filenames', 'stripped_OID',
    'exact_date', 'broadcast_date', 'cls_ID', 'OID', 'login',
    'sequence', 'modified_by', 'modified_on', 'work_duration',
    'work_duration_compl', 'spt_filenames'
}

#### Try to repreat the approach with the character "�"

Claude's approach fixed the problems with the character "?" but another character is there.

However there are still quite a bit of problems with the fix. I found more examples which had erroneous corrections but much more probably exist.
What we did was the best we could.

In [48]:
SECOND_MANUAL_MAP = {k.replace("?", "�"): v for k,v in MANUAL_MAP.items()}
# add examples by hand from errors remaining spotted
SECOND_MANUAL_MAP["Ch�teaureynaud"] = "Châteaureynaud"
SECOND_MANUAL_MAP

{'�': 'à',
 'Parl�': 'Parlé',
 "'Parl�": "'Parlé",
 'Clarifi�': 'Clarifié',
 'Valid�': 'Validé',
 "Premi�re'": "Première'",
 'Accept�': 'Accepté',
 'Mont�': 'Monté',
 'Premi�re': 'Première',
 'St�r�o': 'Stéréo',
 "'Pr�sentateur": "'Présentateur",
 'Pr�sentateur': 'Présentateur',
 "'Interview�": "'Interviewé",
 'Interview�': 'Interviewé',
 'Gen�ve': 'Genève',
 "'Gen�ve": "'Genève",
 "Gen�ve'": "Genève'",
 'Radio-Gen�ve': 'Radio-Genève',
 "Radio-Gen�ve'": "Radio-Genève'",
 'Neuch�tel': 'Neuchâtel',
 "'Neuch�tel": "'Neuchâtel",
 "Neuch�tel'": "Neuchâtel'",
 'neuch�telois': 'neuchâtelois',
 'neuch�teloise': 'neuchâteloise',
 'B�le': 'Bâle',
 'Del�mont': 'Delémont',
 'Z�rich': 'Zürich',
 'Montr�al': 'Montréal',
 'J�rusalem': 'Jérusalem',
 "J�rusalem'": "Jérusalem'",
 'T�h�ran': 'Téhéran',
 'Qu�bec': 'Québec',
 'Alg�rie': 'Algérie',
 "'Alg�rie'": "'Algérie'",
 "d'Alg�rie": "d'Algérie",
 "l'Alg�rie": "l'Algérie",
 'P�kin': 'Pékin',
 'Cor�e': 'Corée',
 'Ha�ti': 'Haïti',
 'Za�re': 'Zaïre',
 'Is

In [ ]:
df_to_fix_path = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata.rts_v3.csv"
df_fixed_path_int = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts_fixed_int.csv"
df_fixed_path = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts_fixed.csv"

df_to_fix = pd.read_csv(df_to_fix_path)

TEXT_COLS = [c for c in df_to_fix.columns if c not in ID_COLS]

print("Processing cells for char = '?' ...")
total_fixes = 0
for col in TEXT_COLS:
    before = df_to_fix[col].copy()
    df_to_fix[col] = df_to_fix[col].apply(fix_cell)
    fixes = (before != df_to_fix[col]).sum()
    total_fixes += fixes
    if fixes > 0:
        print(f"  {col}: {fixes} cells modified")

print(f"\nTotal cells modified: {total_fixes}")

# Verify key rows
print("\n--- Sample verification ---")
check_cols = ['broadcast_episode_title', 'content_summary', 'workflow_status', 'document_type', 'modulation_type']
for col in check_cols:
    samples = df_to_fix[col].dropna().head(5).tolist()
    print(f"\n{col}:")
    for s in samples[:3]:
        print(f"  {str(s)[:120]}")


print("\nSaving...")
df_to_fix.to_csv(df_fixed_path_int, index=False)


print("Processing cells for char = '�' ...")
total_fixes = 0
for col in TEXT_COLS:
    before = df_to_fix[col].copy()
    df_to_fix[col] = df_to_fix[col].apply(fix_cell, char='�', _map=SECOND_MANUAL_MAP)
    fixes = (before != df_to_fix[col]).sum()
    total_fixes += fixes
    if fixes > 0:
        print(f"  {col}: {fixes} cells modified")

print(f"\nTotal cells modified: {total_fixes}")

# Verify key rows
print("\n--- Sample verification ---")
check_cols = ['broadcast_episode_title', 'content_summary', 'workflow_status', 'document_type', 'modulation_type']
for col in check_cols:
    samples = df_to_fix[col].dropna().head(5).tolist()
    print(f"\n{col}:")
    for s in samples[:3]:
        print(f"  {str(s)[:120]}")

print("\nSaving...")
df_to_fix.to_csv(df_fixed_path, index=False)
print("Done!")

Processing cells for char = '?' ...
  broadcast_episode_title: 19134 cells modified
  broadcast_program_name: 1549 cells modified
  document_type: 20956 cells modified
  physical_support_history: 3576 cells modified
  production_type: 521 cells modified
  recording_place: 19605 cells modified
  rights_notes: 22818 cells modified
  rights_status: 22332 cells modified
  series_title: 8917 cells modified
  content_summary: 21474 cells modified
  workflow_status: 20956 cells modified
  assembly_status: 22585 cells modified
  live: 670 cells modified
  modulation_type: 4832 cells modified
  geographical_descriptors: 15692 cells modified
  person_descriptors: 16951 cells modified
  thematical_descriptors: 18174 cells modified
  rights_usage_possibilities: 8538 cells modified
  radio_channels: 15075 cells modified
  subdomains: 10830 cells modified
  recording_dates: 492 cells modified
  first_broadcast_dates: 1056 cells modified
  supports: 16948 cells modified
  participants: 17654 cells mo

## Perform some additional post-processing and cleaning 

This is specifically to separate the case for the metadata and issue index creation

In [3]:
rts_metadata_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts.csv"

rts_metadata_df = pd.read_csv(rts_metadata_filepath)
print(len(rts_metadata_df))
rts_metadata_df.head()

26261


,alias,date_str,stt_filename,mp3_filenames,stripped_OID,exact_date,broadcast_date,cls_ID,OID,login,...,person_descriptors,thematical_descriptors,rights_usage_possibilities,radio_channels,subdomains,recording_dates,first_broadcast_dates,supports,spt_filenames,participants
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,['677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_05...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},NaN,...,"['X-Files', 'Aux frontières du réel']","['série télèvisée', 'science-fiction']",NaN,['Espace 2'],['Interview'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, '7UBM_057338_1{8cfcf300-a815-4f92-bea2-...","[{'name': 'Frias, Roxanne', 'function': 'Inter..."
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,['640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_05...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},NaN,...,NaN,"['école primaire', 'Internet', 'technique péda...",NaN,['Espace 2'],"['Commentaire', 'Parlé divers']",['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['1BBM_058066_6{2782ba11-7050-44d3-bef1-a81252...,"[{'name': 'Dubois, Laurent', 'function': 'Inte..."
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,['2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_05...,2AF13E50-C6A2-49E5-9851-06C441833B50,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{2AF13E50-C6A2-49E5-9851-06C441833B50},NaN,...,"['SSR', 'TSR']","['censure', 'télévision', 'morale']",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, 'RPBM_057474_1{8116d0d3-1c12-44cf-a8a2-...","[{'name': 'Duparc, Nicole', 'function': 'Inter..."
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,['c940bd19-5503-4918-852b-d40c1c18b359_O1BM_05...,C940BD19-5503-4918-852B-D40C1C18B359,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{C940BD19-5503-4918-852B-D40C1C18B359},NaN,...,NaN,"['série télèvisée', 'urgence médicale', 'télés...",NaN,['Espace 2'],['Interview'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['O1BM_057153_2{396b576c-7ed9-4018-a251-c12a84...,"[{'name': 'Kiefer, B.', 'function': 'Interview..."
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,['6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_05...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},NaN,...,NaN,"['enseignement secondaire', 'technique pédagog...",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['FHBM_055791_2{e8455981-078d-44f2-bb94-2e22cd...,"[{'name': 'Prophan, Geneviève', 'function': 'I..."


### Columns filtering 

There was originally 36856 documents in the metadata, 10659 of which were missing MP3 and/or XML files (26261 had both). 

First remove the unused columns in this context, as well as all broadcasts which have no dates.

There are two main uses: the metadata, and the issue index. The filtering for the issue index columns will take place later on

In [4]:
rts_metadata_df.columns

Index(['alias', 'date_str', 'stt_filename', 'mp3_filenames', 'stripped_OID',
       'exact_date', 'broadcast_date', 'cls_ID', 'OID', 'login',
       'broadcast_episode_title', 'sequence', 'broadcast_program_name',
       'document_type', 'hierarchy_level', 'modified_by', 'modified_on',
       'physical_support_history', 'production_type', 'recording_place',
       'rights_notes', 'rights_status', 'series_title', 'content_summary',
       'workflow_status', 'assembly_status', 'live', 'modulation_type',
       'work_duration', 'work_duration_compl', 'geographical_descriptors',
       'person_descriptors', 'thematical_descriptors',
       'rights_usage_possibilities', 'radio_channels', 'subdomains',
       'recording_dates', 'first_broadcast_dates', 'supports', 'spt_filenames',
       'participants'],
      dtype='str')

In [5]:
useful_metadata_cols = ['alias', 'date_str', 'stt_filename', 'mp3_filenames', 'stripped_OID', 'OID', 'exact_date', 'broadcast_date', 
                             'broadcast_episode_title', 'broadcast_program_name',
                             'recording_place', 'series_title', 'content_summary', 'live', 
                             'work_duration', 'participants', 'radio_channels', "subdomains",
                             'recording_dates', 'first_broadcast_dates']

# remove all the broadcasts without a date as we cannot process them
valid_rts_df = rts_metadata_df[~rts_metadata_df['date_str'].isna()]
valid_rts_df = valid_rts_df[useful_metadata_cols]
print(len(valid_rts_df))
valid_rts_df.head()

26225


,alias,date_str,stt_filename,mp3_filenames,stripped_OID,OID,exact_date,broadcast_date,broadcast_episode_title,broadcast_program_name,recording_place,series_title,content_summary,live,work_duration,participants,radio_channels,subdomains,recording_dates,first_broadcast_dates
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,['677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_05...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},True,True,"""X-Files"" ou ""Aux frontières du réel""",Analyse des médias,NaN,NaN,"Présentation de la série-culte. Le succès, com...",Non live,00:00:00.000,"[{'name': 'Frias, Roxanne', 'function': 'Inter...",['Espace 2'],['Interview'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996']
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,['640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_05...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},True,True,Internet à l'Ecole primaire d'Avully (GE),Analyse des médias,Avully,NaN,NaN,Non live,00:00:00.000,"[{'name': 'Dubois, Laurent', 'function': 'Inte...",['Espace 2'],"['Commentaire', 'Parlé divers']",['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997']
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,['2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_05...,2AF13E50-C6A2-49E5-9851-06C441833B50,{2AF13E50-C6A2-49E5-9851-06C441833B50},True,True,Le carré blanc,Analyse des médias,Genève;Fribourg,NaN,NaN,Non live,00:00:00.000,"[{'name': 'Duparc, Nicole', 'function': 'Inter...",['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997']
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,['c940bd19-5503-4918-852b-d40c1c18b359_O1BM_05...,C940BD19-5503-4918-852B-D40C1C18B359,{C940BD19-5503-4918-852B-D40C1C18B359},True,True,"Le feuilleton télévisé ""Urgences""",Analyse des médias,Genève,NaN,Analyse de la série et des raisons de son succès.,Non live,00:00:00.000,"[{'name': 'Kiefer, B.', 'function': 'Interview...",['Espace 2'],['Interview'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996']
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,['6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_05...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},True,True,Le journal à l'école,Analyse des médias,Genève;Fribourg,NaN,NaN,Non live,00:00:00.000,"[{'name': 'Prophan, Geneviève', 'function': 'I...",['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997']


### Choose the correct audio when we have multiple

Finally we removed 36 broadcasts which were missing their date, ending up with 26225 radio issues.

We are ready to assign an issue ID to each broadcast.

We need to check which broadcasts have more than one mp3_filename associated to it. 

Remove any potential duplicated values and ensure that all of the values of "mp3_filenames" actually correspond to a file

In [6]:
# first ensure that column 'mp3_filenames' contains actual lists without repeated values
valid_rts_df['mp3_filenames'] = valid_rts_df['mp3_filenames'].apply(lambda x: list(set(literal_eval(x) if isinstance(x, str) else x)))

valid_rts_df['correct_mp3_filenames'] = valid_rts_df[['mp3_filenames', "stripped_OID"]].apply(lambda x: [f for f in x.mp3_filenames if x.stripped_OID.lower() in f], axis=1)

valid_rts_df["num_files"] = valid_rts_df['correct_mp3_filenames'].apply(lambda x: len(x))

no_existing_mp3s = valid_rts_df[valid_rts_df["num_files"]==0]
print(f"{len(no_existing_mp3s)} radio recordings have no MP3 file corresponding to what we gathered")

multiple_mp3s = valid_rts_df[valid_rts_df["num_files"]>1]
print(f"{len(multiple_mp3s)} radio recordings have more than 1 MP3 file corresponding to what we gathered")

0 radio recordings have no MP3 file corresponding to what we gathered
19 radio recordings have more than 1 MP3 file corresponding to what we gathered


There are 19 broadcasts which have more than one MP3 file, we can prform a manual check for them and create a mapping from OID to correct MP3 filename

In [62]:
multiple_mp3s

,alias,date_str,stt_filename,mp3_filenames,stripped_OID,OID,exact_date,broadcast_date,broadcast_episode_title,broadcast_program_name,...,content_summary,live,work_duration,participants,radio_channels,subdomains,recording_dates,first_broadcast_dates,correct_mp3_filenames,num_files
117,culte,31/01/1993,6E9E7101-CDD9-4480-954B-710D178DC4FE_STT.xml,[6e9e7101-cdd9-4480-954b-710d178dc4fe_Bd12183....,6E9E7101-CDD9-4480-954B-710D178DC4FE,{6E9E7101-CDD9-4480-954B-710D178DC4FE},True,True,Culte à la Chapelle de l'Institut des Diacones...,Culte,...,NaN,Live,00:00:00.000,"[{'name': 'Ruegg, Ulrich', 'function': ""Auteur...",['Espace 2'],['Service religieux'],['31/01/1993 - 31/01/1993'],['31/01/1993 - __/__/____'],[6e9e7101-cdd9-4480-954b-710d178dc4fe_Bd12183....,2
119,culte,03/04/1994,F410C05A-F150-449F-B237-E855F49E1A14_STT.xml,[f410c05a-f150-449f-b237-e855f49e1a14_Bd13190....,F410C05A-F150-449F-B237-E855F49E1A14,{F410C05A-F150-449F-B237-E855F49E1A14},True,True,Culte à la Chapelle romande de Thoune : Prédic...,Culte,...,NaN,Live,00:00:00.000,"[{'name': 'Hollenweger, Walter', 'function': ""...",['Espace 2'],['Service religieux'],['03/04/1994 - 03/04/1994'],['03/04/1994 - __/__/____'],[f410c05a-f150-449f-b237-e855f49e1a14_Bd13190....,2
121,culte,11/12/1994,8A6E2543-E520-4EB6-8DF3-610434D3F631_STT.xml,[8a6e2543-e520-4eb6-8df3-610434d3f631_D5BM_060...,8A6E2543-E520-4EB6-8DF3-610434D3F631,{8A6E2543-E520-4EB6-8DF3-610434D3F631},True,True,Culte à la collégiale de Neuchâtel : Prédicati...,Culte,...,NaN,Live,00:00:00.000,"[{'name': 'Burkhalter, Carmen', 'function': ""A...",['Espace 2'],['Service religieux'],['11/12/1994 - 11/12/1994'],['11/12/1994 - __/__/____'],[8a6e2543-e520-4eb6-8df3-610434d3f631_D5BM_060...,2
150,culte,07/07/1991,88F17839-E0BA-4467-8B12-841D3517BF41_STT.xml,[88f17839-e0ba-4467-8b12-841d3517bf41_R7BM_060...,88F17839-E0BA-4467-8B12-841D3517BF41,{88F17839-E0BA-4467-8B12-841D3517BF41},True,True,Culte au Temple de l'Isle (VD) : Prédication d...,Culte,...,NaN,Live,00:00:00.000,"[{'name': 'Kobi, Georges', 'function': ""Auteur...",['La Première'],['Service religieux'],['07/07/1991 - 07/07/1991'],['07/07/1991 - __/__/____'],[88f17839-e0ba-4467-8b12-841d3517bf41_R7BM_060...,2
193,culte,18/10/1992,138A9D86-E361-4C34-913C-50C83AD3E502_STT.xml,[138a9d86-e361-4c34-913c-50c83ad3e502_Bd11965....,138A9D86-E361-4C34-913C-50C83AD3E502,{138A9D86-E361-4C34-913C-50C83AD3E502},True,True,Culte de l'Eglise Evangélique Libre de Genève ...,Culte,...,NaN,Live,00:00:00.000,"[{'name': 'Sandoz, Marc-Henri', 'function': ""A...",['Espace 2'],['Service religieux'],['18/10/1992 - 18/10/1992'],['18/10/1992 - __/__/____'],[138a9d86-e361-4c34-913c-50c83ad3e502_Bd11965....,2
370,ecoute_paix,01/06/1947,5BF47ED2-2B35-430F-B74B-12884F25292E_STT.xml,[5bf47ed2-2b35-430f-b74b-12884f25292e_BD2948_P...,5BF47ED2-2B35-430F-B74B-12884F25292E,{5BF47ED2-2B35-430F-B74B-12884F25292E},False,True,Avant la Conférence de Paris : L'opinion des m...,A l'écoute de la paix qui vient,...,Commentaires sur le relèvement économique de l...,Non live,00:00:00.000,"[{'name': 'Ladame, Paul', 'function': 'Comment...",['Radio Sottens'],['Commentaire'],['10/06/1947 - 10/06/1947'],['~__/06/1947 - __/__/____'],[5bf47ed2-2b35-430f-b74b-12884f25292e_BD2948_P...,2
5029,j_mat,13/06/1996,70869199-33C8-4200-B59F-A419FBF76B6F_STT.xml,[70869199-33c8-4200-b59f-a419fbf76b6f_3YBM_137...,70869199-33C8-4200-B59F-A419FBF76B6F,{70869199-33C8-4200-B59F-A419FBF76B6F},True,True,Affaire de l'OTS (Ordre du Temple solaire). La...,Journal du matin,...,NaN,Non live,00:00:00.000,"[{'name': 'Lorans, Jean-François', 'function':...",['La Première'],['Interview'],['13/06/1996 - 13/06/1996'],['13/06/1996 - 13/06/1996'],[70869199-33c8-4200-b59f-a419fbf76b6f_3YBM_137...,2
8092,j_midi,01/01/1970,F426F0F0-57AC-4494-8831-0DA9FDBF1A21_STT.xml,[f426f0f0-57ac-4494-8831-0da9fdbf1a21_24684D{2...,F426F0F0-57AC-4494-8831-0DA9FDBF1A21,{F426F0F0-57AC-4494-8831-0DA9FDBF1A21},False,True,Interview de Maurice Chappaz et Jean-Marc Lovay,Journal de 

In [7]:
multiple_mp3s["mp3_filenames"].values

array([list(['6e9e7101-cdd9-4480-954b-710d178dc4fe_Bd12183.01{C0DEE3E8-13F0-4C2E-B6E6-7C26C63BAB5E}.mp3', '6e9e7101-cdd9-4480-954b-710d178dc4fe_ZUBM_060814_2{36d772cc-db58-42de-9935-45fb2c8264fb}.mp3']),
       list(['f410c05a-f150-449f-b237-e855f49e1a14_Bd13190.03{B91FDB78-93BE-4568-89BB-B45FB5BFD2A7}.mp3', 'f410c05a-f150-449f-b237-e855f49e1a14_02BM_061727_3{dc6a1714-a381-46bb-bf3c-c4a6e750dd9a}.mp3']),
       list(['8a6e2543-e520-4eb6-8df3-610434d3f631_D5BM_060586_4{13878992-3dc3-4388-9b21-3794d26fae38}.mp3', '8a6e2543-e520-4eb6-8df3-610434d3f631_Bd11950.03{372CBEB6-8499-4F04-8102-A957660155CD}.mp3']),
       list(['88f17839-e0ba-4467-8b12-841d3517bf41_Bd12317.01{DF589FDD-9A4E-45C4-A14E-5EAD1BC61018}.mp3', '88f17839-e0ba-4467-8b12-841d3517bf41_R7BM_060947_5{bd6e90b3-c5a3-4ed8-a0ef-44a21b6cfe47}.mp3']),
       list(['138a9d86-e361-4c34-913c-50c83ad3e502_7EBM_060600_1{1d86f1f3-0bb8-44f8-9e30-a4a409629c21}.mp3', '138a9d86-e361-4c34-913c-50c83ad3e502_Bd11965.03{996A1D64-CF29-4CA2-8C6A-80

In [7]:
oid_to_mp3 = {
    "6E9E7101-CDD9-4480-954B-710D178DC4FE": "6e9e7101-cdd9-4480-954b-710d178dc4fe_Bd12183.01{C0DEE3E8-13F0-4C2E-B6E6-7C26C63BAB5E}.mp3",
    "F410C05A-F150-449F-B237-E855F49E1A14": "f410c05a-f150-449f-b237-e855f49e1a14_02BM_061727_3{dc6a1714-a381-46bb-bf3c-c4a6e750dd9a}.mp3",
    "8A6E2543-E520-4EB6-8DF3-610434D3F631": "8a6e2543-e520-4eb6-8df3-610434d3f631_D5BM_060586_4{13878992-3dc3-4388-9b21-3794d26fae38}.mp3", 
    "88F17839-E0BA-4467-8B12-841D3517BF41": "88f17839-e0ba-4467-8b12-841d3517bf41_R7BM_060947_5{bd6e90b3-c5a3-4ed8-a0ef-44a21b6cfe47}.mp3",
    "138A9D86-E361-4C34-913C-50C83AD3E502": "138a9d86-e361-4c34-913c-50c83ad3e502_7EBM_060600_1{1d86f1f3-0bb8-44f8-9e30-a4a409629c21}.mp3",
    "5BF47ED2-2B35-430F-B74B-12884F25292E": "5bf47ed2-2b35-430f-b74b-12884f25292e_DJBM_052511_7{bf2c30ff-6d6c-499a-be18-3da690cca1dc}.mp3",
    "70869199-33C8-4200-B59F-A419FBF76B6F": "70869199-33c8-4200-b59f-a419fbf76b6f_3YBM_137898_3{78d213db-2b24-4b88-a6bc-1ad51fc8ee5c}.mp3",
    "F426F0F0-57AC-4494-8831-0DA9FDBF1A21": "f426f0f0-57ac-4494-8831-0da9fdbf1a21_24684D{256CA945-1D64-40BA-9415-E0A5D877F290}.mp3",
    "03CD8CEA-19D3-4DC5-A77E-48A0341C2F0E": "03cd8cea-19d3-4dc5-a77e-48a0341c2f0e_35920(1){60EFC20B-89CA-4484-A253-C857984ED775}.mp3",
    "EDDDFA4D-435B-4DF3-B6BF-291E572EFCC2": "edddfa4d-435b-4df3-b6bf-291e572efcc2_Bd 9824{7E485936-1692-485B-AFD4-360A60CFF5AA}.mp3",
    "029ABC0A-A44A-4147-A0FA-F8C7E908F05F": "029abc0a-a44a-4147-a0fa-f8c7e908f05f_A13891.23_ST{A2963C39-579C-43F3-815B-B76FCBF3B6EA}.mp3",
    "296B9080-AFF5-42BE-A875-61EDC7430635": "296b9080-aff5-42be-a875-61edc7430635_25534(5){113CD020-465B-4306-96FB-6AB46F36D48F}.mp3",
    "B9E0F0DC-B702-48FC-A8B8-A48B633601FF": "b9e0f0dc-b702-48fc-a8b8-a48b633601ff_DSBM_061775_6{e0f91c9e-5daa-4e5f-8a02-8d0d5e50cbad}.mp3",
    "95E82673-3D68-4544-808A-C717670853E8": "95e82673-3d68-4544-808a-c717670853e8_19612{EC371FF6-00A8-45C8-9973-FB2084A6482E}.mp3",
    "A33BD326-E45E-4FA2-BC91-AF78E6D742D6": "a33bd326-e45e-4fa2-bc91-af78e6d742d6_36326{71760BF0-9C8C-409A-A26A-407C6451E079}.mp3",
    "A7F806C4-F18F-434D-B34F-DEE99BA5E861": "a7f806c4-f18f-434d-b34f-dee99ba5e861_3SBM_036801_8{edf84209-829f-418c-ae72-fb3b86ef3b45}.mp3",
    "E19EEF12-8617-4E34-A02B-C21991FCEFEE": "e19eef12-8617-4e34-a02b-c21991fcefee_3RBM_093523_1{4015411f-9d37-471a-9198-b3d79a39bc44}.mp3",
    "8F89CBEF-4225-4B07-A770-CF5146D90BA3": "8f89cbef-4225-4b07-a770-cf5146d90ba3_8OBM_093573_4{b213538f-4637-454e-b7a3-ae5736a6cf6c}.mp3",
    "E6E69150-5115-4782-8DD7-2C525F2009FD": "e6e69150-5115-4782-8dd7-2c525f2009fd_2888{73EC7DD1-5506-42CE-BE03-34EC686C2022}.mp3",
}

In [9]:
valid_rts_df['final_mp3_filename'] = valid_rts_df[['correct_mp3_filenames', 'stripped_OID']].apply(lambda x: x.correct_mp3_filenames[0] if len(x.correct_mp3_filenames)==1 else oid_to_mp3[x.stripped_OID], axis=1)

valid_rts_df

,alias,date_str,stt_filename,mp3_filenames,stripped_OID,OID,exact_date,broadcast_date,broadcast_episode_title,broadcast_program_name,...,live,work_duration,participants,radio_channels,subdomains,recording_dates,first_broadcast_dates,correct_mp3_filenames,num_files,final_mp3_filename
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,[677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_057...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},True,True,"""X-Files"" ou ""Aux frontières du réel""",Analyse des médias,...,Non live,00:00:00.000,"[{'name': 'Frias, Roxanne', 'function': 'Inter...",['Espace 2'],['Interview'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996'],[677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_057...,1,677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_0573...
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,[640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_058...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},True,True,Internet à l'Ecole primaire d'Avully (GE),Analyse des médias,...,Non live,00:00:00.000,"[{'name': 'Dubois, Laurent', 'function': 'Inte...",['Espace 2'],"['Commentaire', 'Parlé divers']",['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997'],[640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_058...,1,640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_0580...
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,[2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_057...,2AF13E50-C6A2-49E5-9851-06C441833B50,{2AF13E50-C6A2-49E5-9851-06C441833B50},True,True,Le carré blanc,Analyse des médias,...,Non live,00:00:00.000,"[{'name': 'Duparc, Nicole', 'function': 'Inter...",['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997'],[2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_057...,1,2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_0574...
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,[c940bd19-5503-4918-852b-d40c1c18b359_O1BM_057...,C940BD19-5503-4918-852B-D40C1C18B359,{C940BD19-5503-4918-852B-D40C1C18B359},True,True,"Le feuilleton télévisé ""Urgences""",Analyse des médias,...,Non live,00:00:00.000,"[{'name': 'Kiefer, B.', 'function': 'Interview...",['Espace 2'],['Interview'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996'],[c940bd19-5503-4918-852b-d40c1c18b359_O1BM_057...,1,c940bd19-5503-4918-852b-d40c1c18b359_O1BM_0571...
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,[6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_055...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},True,True,Le journal à l'école,Analyse des médias,...,Non live,00:00:00.000,"[{'name': 'Prophan, Geneviève', 'function': 'I...",['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997'],[6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_055...,1,6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_0557...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26256,vie_va,12/11/1984,B36F7349-DA77-412C-A42D-F8C7A5FF8E61_STT.xml,[b36f7349-da77-412c-a42d-f8c7a5ff8e61_34180.01...,B36F7349-DA77-412C-A42D-F8C7A5FF8E61,{B36F7349-DA77-412C-A42D-F8C7A5FF8E61},True,True,Un poète. 3 illustrations : Interview de Jean ...,Vie qui va,...,Non live,00:00:00.000,"[{'name': 'Pache, Jean', 'function': 'Intervie...",['Espace 2'],['Interview'],['12/11/1984 - 12/11/1984 / Avant'],['12/11/1984 - 12/11/1984'],[b36f7349-da77-412c-a42d-f8c7a5ff8e61_34180.01...,1,b36f7349-da77-412c-a42d-f8c7a5ff8e61_34180.01{...
26257,vie_va,07/11/1984,DC22DFF9-49DA-42C0-909B-55B9E4B78D33_STT.xml,[dc22dff9-49da-42c0-909b-55b9e4b78d33_34178{7E...,DC22DFF9-49DA-42C0-909B-55B9E4B78D33,{DC22DFF9-49DA-42C0-909B-55B9E4B78D33},True,True,Une Suissesse au Canada : Entretien avec Lça P...,Vie qui va,...,Non live,00:00:00.000,"[{'name': 'Pool, Lça', 'function': 'Interviewé...",['Espace 2'],['Interview'],['07/11/1984 - 07/11/1984 / Avant'],['07/11/1984 - __/__/____'],[dc22dff9-49da-42c0-909b-55b9e4b78d3

### Save the current version (to create index, and to resume for metadata prep)

In [10]:
metadata_prep_filename = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/unique_programs_metadata.rts.csv"

valid_rts_df.to_csv(metadata_prep_filename, index=False)

# Finalize the processing into a file which is used during ingestion

In [99]:
rts_metadata_filepath = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/programs_metadata_rts.csv"

rts_metadata_df = pd.read_csv(rts_metadata_filepath)
print(len(rts_metadata_df))
rts_metadata_df.head()

26261


,alias,date_str,stt_filename,mp3_filenames,stripped_OID,exact_date,broadcast_date,cls_ID,OID,login,...,person_descriptors,thematical_descriptors,rights_usage_possibilities,radio_channels,subdomains,recording_dates,first_broadcast_dates,supports,spt_filenames,participants
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,['677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_05...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},NaN,...,"['X-Files', 'Aux frontières du réel']","['série télèvisée', 'science-fiction']",NaN,['Espace 2'],['Interview'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, '7UBM_057338_1{8cfcf300-a815-4f92-bea2-...","[{'name': 'Frias, Roxanne', 'function': 'Inter..."
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,['640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_05...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},NaN,...,NaN,"['école primaire', 'Internet', 'technique péda...",NaN,['Espace 2'],"['Commentaire', 'Parlé divers']",['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['1BBM_058066_6{2782ba11-7050-44d3-bef1-a81252...,"[{'name': 'Dubois, Laurent', 'function': 'Inte..."
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,['2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_05...,2AF13E50-C6A2-49E5-9851-06C441833B50,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{2AF13E50-C6A2-49E5-9851-06C441833B50},NaN,...,"['SSR', 'TSR']","['censure', 'télévision', 'morale']",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, 'RPBM_057474_1{8116d0d3-1c12-44cf-a8a2-...","[{'name': 'Duparc, Nicole', 'function': 'Inter..."
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,['c940bd19-5503-4918-852b-d40c1c18b359_O1BM_05...,C940BD19-5503-4918-852B-D40C1C18B359,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{C940BD19-5503-4918-852B-D40C1C18B359},NaN,...,NaN,"['série télèvisée', 'urgence médicale', 'télés...",NaN,['Espace 2'],['Interview'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['O1BM_057153_2{396b576c-7ed9-4018-a251-c12a84...,"[{'name': 'Kiefer, B.', 'function': 'Interview..."
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,['6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_05...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},NaN,...,NaN,"['enseignement secondaire', 'technique pédagog...",NaN,['Espace 2'],['Interview'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['FHBM_055791_2{e8455981-078d-44f2-bb94-2e22cd...,"[{'name': 'Prophan, Geneviève', 'function': 'I..."


Start by removing duplicated values

In [100]:
duplicated = rts_metadata_df[rts_metadata_df.duplicated()]
print(f"There are {len(duplicated)} duplicated values in this data!, in the aliases {set(duplicated.alias.values)}")

There are 64 duplicated values in this data!, in the aliases {'ombres_eco', 'mag_eco'}


In [101]:
rts_metadata_df = rts_metadata_df.drop_duplicates()

print(f"There are {len(rts_metadata_df)} non-duplicated records.")

There are 26197 non-duplicated records.


## First get the list of possible broadcast show types using the column "subdomains"

Currently, the column is formatted as string which represents a list of values. We want to have the full list of possible values to extract a small set of possible types

In [102]:
first_type_mapping = {
    "COM": "Commentaire",
    "DEB": "Débat",
    "DECL": "Déclaration",
    "DIV": "Parlé divers",
    "INT": "Interview",
    "REP": "Reportage",
    "VAR": "Variétés",
    "Thôétre": "Théâtre"
}

This the full list of possible types we have. Some are abbreviations of others, and they can in general be grouped according to their subject or theme.

First, 'Thôétre' should be 'Théâtre', and some 

Then we have:
COM -> Commentaire
DEB -> Débat
DECL -> Déclaration
DIV -> Divers
INT -> Interview
REP -> Reportage
VAR -> Variétés

In [103]:
# first do a literal eval on the values to actually get lists of values.

rts_metadata_df['broadcast_types'] = rts_metadata_df['subdomains'].apply(lambda x: literal_eval(x) if isinstance(x, str) else [])

possible_types = Counter([first_type_mapping[t] if t in first_type_mapping else t for v in rts_metadata_df['broadcast_types'].values for t in v ])
possible_types

Counter({'Interview': 12357,
         'Commentaire': 4462,
         'Déclaration': 1094,
         'Reportage': 458,
         'Documentaire': 281,
         'Parlé divers': 143,
         'Débat': 130,
         'Service religieux': 127,
         'Récit': 44,
         'Conférence': 20,
         'Animation': 13,
         'Variétés': 9,
         'Théâtre': 5,
         'Sketch': 3,
         'Jeu radiophonique': 3,
         'Indicatif': 3})

We got Claude to translate these types.

Here are some of its comments:
- `Indicatif`: in French radio terminology, indicatif specifically refers to the short musical signature used to identify a programme — so "theme tune" or "jingle" is the standard translation. It's distinct from a full musical piece; it's the brief recurring sonic identifier of a show.
That said, if you're working in a more technical broadcasting context, "station ID" or "programme signature" might be more precise alternatives.
- `Parlé divers` — a catch-all for spoken content that doesn't fit other categories, hence "miscellaneous speech."
- `Animation` — in French broadcasting, animation refers to the hosting/presenting role that keeps a programme flowing, not animation in the visual/cartoon sense.
- `Conférence` — can be either a formal lecture or a conference address, so both are given.
- `Théâtre` — in a radio context this is a performed dramatic piece, equivalent to a radio play.

The idea will then to map all these classes to three different very "wide" classes for the existing "tp" field of the content-item metadata. 
In this context, We want to map all classes to one of "article" (general news), "discussion" (for all interview/exchange related content) and "entertainement". 

We will also ask claude for this mapping for bith RTS and INA (to ensure a relativel coherent mapping), but this will take place during canonical ingestion.

In [104]:
# asked Claude to translate them to english, and did a mini normalization
rts_broadcast_types_en = {
    'Interview': 'Interview',
    'Commentaire': 'Commentary',
    'Déclaration': 'Statement',
    'Reportage': 'Report',
    'Documentaire': 'Documentary',
    'Parlé divers': 'Miscellaneous speech',
    'Débat': 'Debate',
    'Service religieux': 'Religious service',
    'Récit': 'Narrative',
    'Conférence': 'Conference', # 'Lecture / Conference',
    'Animation': 'Entertainment', # 'Hosting / Entertainment',
    'Variétés': 'Variety show', 
    'Théâtre': 'Theatre', #'Theatre / Radio play',
    'Sketch': 'Sketch',
    'Jeu radiophonique': 'Radio game show',
    'Indicatif': 'Jingle', # 'Theme tune / Jingle',
}


In [105]:

rts_broadcast_en_norm_types = {
    'Interview':        ('Interview',               'discussion'),
    'Commentaire':      ('Commentary',              'article'),
    'Déclaration':      ('Statement',               'article'),
    'Reportage':        ('Report',                  'article'),
    'Documentaire':     ('Documentary',             'article'),
    'Parlé divers':     ('Miscellaneous speech',    'article'),
    'Débat':            ('Debate',                  'discussion'),
    'Service religieux':('Religious service',       'chronicle'),
    'Récit':            ('Narrative',               'article'),
    'Conférence':       ('Conference',              'article'),
    'Animation':        ('Hosting',                 'entertainment'),
    'Variétés':         ('Variety show',            'entertainment'),
    'Théâtre':          ('Theatre',                 'entertainment'),
    'Sketch':           ('Sketch',                  'entertainment'),
    'Jeu radiophonique':('Radio game show',         'entertainment'),
    'Indicatif':        ('Jingle',                  'entertainment'),
    # re-add the first types prior to reformatting
    "COM":              ('Commentary',              'article'),
    "DEB":              ('Debate',                  'discussion'),
    "DECL":             ('Statement',               'article'),
    "DIV":              ('Miscellaneous speech',    'article'),
    "INT":              ('Interview',               'discussion'),
    "REP":              ('Report',                  'article'),
    "VAR":              ('Variety show',            'entertainment'),
}

### Having the same approach for INA

Then we also want to collect the possible broadcast types of INA to have a comprehensive view of the possibilities.

We also translate them:
- `Papier` — in broadcast journalism, a papier is a prepared scripted piece read by a journalist, closest to "news piece" or "copy."
- `Bruitage image sonore` — refers to sound design/effects used to accompany imagery, hence "sound effects / audio image."
- `Elément brut` — raw, unedited material, i.e. "raw footage."
- `Micro trottoir` — the French term for street interviews / man-on-the-street segments, standardly translated as "vox pop."
- `Indicatif` — the signature tune or jingle identifying a programme, hence "theme tune / jingle."
- `Evocation scénarisée` — a dramatised or scripted reconstruction of events, so "scripted evocation" or "dramatised reconstruction."

In [106]:
ina_broadcast_types = {'Journal parlé': 21416,
         'Magazine': 9454,
         'Reportage': 3897,
         'Causerie': 3787,
         'Interview entretien': 2550,
         'Débat': 2530,
         'Papier': 1718,
         'Déclaration': 983,
         'Interview': 252,
         'Bruitage image sonore': 185,
         'Chronique': 152,
         'Lecture': 96,
         'Interprétation': 94,
         'Récit portrait': 88,
         'Entretien': 88,
         'Elément brut': 30,
         'Retransmission': 28,
         'Documentaire': 27,
         'Réalisation dans un lieu public': 26,
         'Conférence de presse': 14,
         'Evocation scénarisée': 12,
         'Indicatif': 11,
         'Emission à base de disques': 8,
         'Témoignage': 7,
         "Document à base d'archives": 6,
         'Rétrospective': 5,
         'Jeu': 5,
         'Feuilleton': 5,
         'Brève': 3,
         'Micro trottoir': 3,
         'Message info': 1,
         'Création sonore': 1,
         'Revue de presse': 1,
         'Dramatique': 1,
         'Message publicitaire': 1,
         'Best of': 1,
         "Cours d'enseignement": 1,
         'Tranche horaire': 1}

ina_broadcast_types_en = {
    'Journal parlé': 'News bulletin',
    'Magazine': 'Magazine',
    'Reportage': 'Report',
    'Causerie': 'Chat',
    'Interview entretien': 'Interview', # 'Interview / Discussion',
    'Débat': 'Debate',
    'Papier': 'News piece',
    'Déclaration': 'Statement',
    'Interview': 'Interview',
    'Bruitage image sonore': 'Sound effects', # 'Sound effects / Audio image',
    'Chronique': 'Chronicle',
    'Lecture': 'Reading',
    'Interprétation': 'Performance',
    'Récit portrait': 'Profile story',
    'Entretien': 'Discussion',
    'Elément brut': 'Raw footage',
    'Retransmission': 'Replay', #'Re-broadcast / Replay',
    'Documentaire': 'Documentary',
    'Réalisation dans un lieu public': 'Public location recording',
    'Conférence de presse': 'Press conference',
    'Evocation scénarisée': 'Scripted evocation',
    'Indicatif': 'Jingle', # 'Theme tune / Jingle',
    'Emission à base de disques': "Music show", #'Record-based programme',
    'Témoignage': 'Testimony',
    "Document à base d'archives": 'Archive-based document',
    'Rétrospective': 'Retrospective',
    'Jeu': 'Game show',
    'Feuilleton': 'Serial',
    'Brève': 'News brief',
    'Micro trottoir': 'Street interview', #'Vox pop',
    'Message info': 'Information message',
    'Création sonore': 'Sound creation',
    'Revue de presse': 'Press review',
    'Dramatique': "Drama", #'Radio drama',
    'Message publicitaire': 'Advertisement',
    'Best of': 'Best of',
    "Cours d'enseignement": 'Educational lesson',
    'Tranche horaire': 'Time slot',
}

Now the normalized INA types.

In [107]:
ina_broadcast_en_norm_types = {
    'Journal parlé':                    ('News bulletin',           'article'),
    'Magazine':                         ('Magazine',                'article'),
    'Reportage':                        ('Report',                  'article'),
    'Causerie':                         ('Chat',                    'discussion'),
    'Interview entretien':              ('Interview',               'discussion'),
    'Débat':                            ('Debate',                  'discussion'),
    'Papier':                           ('News piece',              'article'),
    'Déclaration':                      ('Statement',               'article'),
    'Interview':                        ('Interview',               'discussion'),
    'Bruitage image sonore':            ('Sound effects',           'entertainment'),
    'Chronique':                        ('Chronicle',               'chronicle'),
    'Lecture':                          ('Reading',                 'entertainment'),
    'Interprétation':                   ('Performance',             'entertainment'),
    'Récit portrait':                   ('Profile story',           'article'),
    'Entretien':                        ('Discussion',              'discussion'),
    'Elément brut':                     ('Raw footage',             'article'),
    'Retransmission':                   ('Replay',                  'article'),
    'Documentaire':                     ('Documentary',             'article'),
    'Réalisation dans un lieu public':  ('Public location recording','article'),
    'Conférence de presse':             ('Press conference',        'discussion'),
    'Evocation scénarisée':             ('Scripted evocation',      'entertainment'),
    'Indicatif':                        ('Jingle',                  'entertainment'),
    'Emission à base de disques':       ('Music show',              'entertainment'),
    'Témoignage':                       ('Testimony',               'discussion'),
    "Document à base d'archives":       ('Archive-based document',  'article'),
    'Rétrospective':                    ('Retrospective',           'article'),
    'Jeu':                              ('Game show',               'entertainment'),
    'Feuilleton':                       ('Serial',                  'entertainment'),
    'Brève':                            ('News brief',              'article'),
    'Micro trottoir':                   ('Street interview',        'discussion'),
    'Message info':                     ('Information message',     'article'),
    'Création sonore':                  ('Sound creation',          'entertainment'),
    'Revue de presse':                  ('Press review',            'article'),
    'Dramatique':                       ('Drama',                   'entertainment'),
    'Message publicitaire':             ('Advertisement',           'ad'),
    'Best of':                          ('Best of',                 'entertainment'),
    "Cours d'enseignement":             ('Educational lesson',      'article'),
    'Tranche horaire':                  ('Time slot',               'article'),
}

#### Now setting the values and formatting the metadata file to allow for canonical ingest

In [ ]:
# First set the translated broadcast types and normalized types 

rts_metadata_df['broadcast_types_en'] = rts_metadata_df['broadcast_types'].apply(lambda x: [rts_broadcast_en_norm_types[y][0] for y in x])
rts_metadata_df['normalized_types'] = rts_metadata_df['broadcast_types'].apply(lambda x: list(set([rts_broadcast_en_norm_types[y][1] for y in x])))

multiple_types = rts_metadata_df['normalized_types'].apply(lambda x: len(x)>1)
print(any(multiple_types))

# Define the priority for types
priority_order = {
    "ad": 0,
    "chronicle": 1,
    "discussion": 2,
    "article": 3,
    "entertainment": 4,
    "chronicle": 5,
}

# Apply to normalized_types column - keeps only the highest priority (lowest number)
rts_metadata_df['normalized_types'] = rts_metadata_df['normalized_types'].apply(
    lambda x: None if len(x)==0 else min(x, key=lambda t: priority_order.get(t, 999))
)

rts_metadata_df.head()

## Now finalize the formatting of other columns and create the final dict

### Cleaning the Radio Channels

Some columns have lists of values for which we need to literal eval: "geographical_descriptors", "person_descriptors", "thematical_descriptors", "participants", "radio_channels".

- "participants" is a dict which needs to be uniformized somehow
- "radio_channels" in turn needs to also be turned to a unique value if possible.

In [ ]:
rts_metadata_df['radio_channels'] = rts_metadata_df['radio_channels'].apply(lambda x: literal_eval(x) if isinstance(x, str) else (x if isinstance(x, list) and len(x)!=0 else []))


rts_metadata_df['radio_channel'] = rts_metadata_df['radio_channels'].apply(lambda x: x[0] if len(x)==1 else (None if x==[] else x))

print(any(rts_metadata_df['radio_channel'].isna()))
multiple_channels = rts_metadata_df['radio_channels'].apply(lambda x: len(x)>1)
print(any(multiple_channels))

rts_metadata_df

In [ ]:
possible_channels = Counter([t for v in rts_metadata_df['radio_channels'].values for t in v ])
possible_channels

Counter({'La Première': 12185,
         'Premier programme': 8050,
         'Deuxième programme': 749,
         'Espace 2': 685,
         'Radio Sottens': 253,
         'Radio-Genève': 71,
         'Radio-Lausanne': 45,
         'Option Musique': 4})

In [111]:
# check the records which have more than one channel:
rts_metadata_df[multiple_channels]

,alias,date_str,stt_filename,mp3_filenames,stripped_OID,exact_date,broadcast_date,cls_ID,OID,login,...,subdomains,recording_dates,first_broadcast_dates,supports,spt_filenames,participants,broadcast_types,broadcast_types_en,normalized_types,radio_channel
657,enquest,23/06/1978,28FC3BE3-EFDB-44FF-AFE1-0707544518FB_STT.xml,['28fc3be3-efdb-44ff-afe1-0707544518fb_MA 7810...,28FC3BE3-EFDB-44FF-AFE1-0707544518FB,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{28FC3BE3-EFDB-44FF-AFE1-0707544518FB},NaN,...,['Interview'],['__/06/1978 - __/06/1978'],['23/06/1978 - __/__/____'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['MA 78105{229CF715-67D6-45E7-ADF0-95506FD2158...,"[{'name': 'Lichtenthaeler, Charles', 'function...",[Interview],[Interview],discussion,"[Radio Sottens, Premier programme]"
18055,mag_info,15/05/1967,BA956CC5-C0AE-4299-9A81-582D179E1DF0_STT.xml,['ba956cc5-c0ae-4299-9a81-582d179e1df0_A11894{...,BA956CC5-C0AE-4299-9A81-582D179E1DF0,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{BA956CC5-C0AE-4299-9A81-582D179E1DF0},NaN,...,['Interview'],['28/04/1967 - 28/04/1967'],"['15/05/1967 - __/__/____', '06/09/1967 - __/_...",[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['A11894{8C18D1ED-FF64-435D-B17B-5CA773BBA932}...,"[{'name': 'Mottu, Philippe', 'function': 'Inte...",[Interview],[Interview],discussion,"[Premier programme, Deuxième programme]"
18947,min_oecu,01/01/1970,F398E482-763A-49A5-A9B9-5A12A3F5684B_STT.xml,['f398e482-763a-49a5-a9b9-5a12a3f5684b_Bd 1429...,F398E482-763A-49A5-A9B9-5A12A3F5684B,False,False,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{F398E482-763A-49A5-A9B9-5A12A3F5684B},NaN,...,"['Indicatif', 'Parlé divers']",['~__/__/1970 - ~__/__/1970'],NaN,[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, 'Bd 14294.01{CE3FD9D6-7F82-48D7-A168-66...","[{'name': '[musique]', 'function': 'Divers', '...","[Indicatif, Parlé divers]","[Jingle, Miscellaneous speech]",article,"[La Première, Espace 2]"
19206,miroir_monde,04/01/1960,A0459940-B832-4C05-B161-CEEFC6E05C38_STT.xml,['a0459940-b832-4c05-b161-ceefc6e05c38_MA 607....,A0459940-B832-4C05-B161-CEEFC6E05C38,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{A0459940-B832-4C05-B161-CEEFC6E05C38},NaN,...,['Interview'],['04/01/1960 - 04/01/1960'],['04/01/1960 - __/__/____'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['MA 607.b{1967BE87-9310-423B-949E-7E0663AA0FC...,"[{'name': 'Inconnu', 'function': 'Interviewé/e...",[Interview],[Interview],discussion,"[Premier programme, Radio Sottens]"
20642,miroir_monde,18/02/1964,B61637F0-56B5-4AB2-BB61-6516863BA9A8_STT.xml,['b61637f0-56b5-4ab2-bb61-6516863ba9a8_A120{3F...,B61637F0-56B5-4AB2-BB61-6516863BA9A8,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{B61637F0-56B5-4AB2-BB61-6516863BA9A8},NaN,...,"['Commentaire', 'Reportage']",['18/02/1964 - 18/02/1964'],['18/02/1964 - __/__/____'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['A120{3FF277E0-189B-43A7-8236-2B26153B6228}.w...,"[{'name': 'Sulser, Christian', 'function': 'Pr...","[Commentaire, Reportage]","[Commentary, Report]",article,"[Radio Sottens, Premier programme]"
21921,miroir_monde,03/07/1973,A2C8B125-5626-4ABB-974C-263793C78AE5_STT.xml,['a2c8b125-5626-4abb-974c-263793c78ae5_27732.6...,A2C8B125-5626-4ABB-974C-263793C78AE5,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{A2C8B125-5626-4ABB-974C-263793C78AE5},NaN,...,['Commentaire'],['03/07/1973 - 03/07/1973'],['03/07/1973 - __/__/____'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['27732.6{5A970BC6-2EAC-4458-9113-2B230E468D08...,"[{'name': 'Karjalainen, Atti [ortographe?]', '...",[Commentaire],[Commentary],article,"[Radio Sottens, Premier programme]"
24860,petitdej,01/07/1996,0CC1CC1C-1929-4BB9-B7D7-07CCD1886866_STT.xml,['0cc1cc1c-1929-4bb9-b7d7-07ccd1886866_826c31c...,0CC1CC1C-1929-4BB9-B7D7-07CCD1886866,False,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{0CC1CC1C-1929-4BB9-B7D7-07CCD1886866},NaN,...,['Interview'],['22/07/1996 - 22/07/1996'],['__/07/1996 - __/07/1996'],[{'spt_clsid': '{9C19C69A-A4DC

I don't exactly know how to proceed with records which have multiple programs. 
There are 8 occasions of this and it's not exactly clear which ones should be chosen or not. 
What I will do is keep both in the provider metadata and choose one in the actual radio channel field, based on some research.

Claude's conclusions and information on this is the following:
Summary table
Name Nature
- La Première✅ Current channel (RTS Première)
- Espace 2✅ Current channel (RTS Espace 2)
- Option Musique✅ Current channel (RTS Option Musique)
- Radio-Genève🕰️ Historical station (defunct since 1974)
- Radio-Lausanne🕰️ Historical station (defunct since 1974)
- Radio Sottens📡 Historical transmitter name, not a channel
- Premier programme📻 Internal programme designation (→ La Première)
- Deuxième programme📻 Internal programme designation (→ Espace 2)


Based on this: 
- we will not choose Radio Sottens when it's in concurrence with another channel.
- Option Musique only occurs 4 times, so we will check when and it it's always on the same program
- When we have "premier programme" and "deuxième programme" we will also check the frequency amongst the show in question and choose accordingly.


The aliases in question are: 
- enquest - ['Radio Sottens', 'Premier programme'] --> "Premier programme"
- mag_info - ['Premier programme', 'Deuxième programme'] --> "Premier programme"
- min_oecu - ['La Première', 'Espace 2'] --> 'La Première'
- miroir_monde - ['Radio Sottens', 'Premier programme'] --> 'Premier programme'
- petitdej - ['Option Musique', 'La Première'] --> 'La Première'
- suisse_euro - ['Premier programme', 'Deuxième programme'] --> 'Premier programme'


Setting this preference was "easy" in these few cases but did provide some insight on values which do seem odd. 
Given that the full list of radio channels will be kept for each broadcast, we can make some modifications for the value atucally going into the "radio_channel" column which will be the one in the metadata field, while the rest will stay in the provider metadata object. 


### Disambiguating for these specific cases

#### a. `enquest`: 'Radio Sottens', 'Premier programme'

In [ ]:
# enquest
enquest_df = rts_metadata_df[rts_metadata_df['alias']=='enquest']

print(enquest_df['radio_channels'].values)
enquest_channels = Counter([t for v in enquest_df['radio_channels'].values for t in v ])

enquest_channels

The counts for enquest are: 
- 'Premier programme': 1045, 
- 'Radio Sottens': 1

Chosen --> 'Premier programme'

#### b. `mag_info`: 'Premier programme', 'Deuxième programme'

In [ ]:
# mag_info
mag_info_df = rts_metadata_df[rts_metadata_df['alias']=='mag_info']

print(mag_info_df['radio_channels'].values)
mag_info_channels = Counter([t for v in mag_info_df['radio_channels'].values for t in v ])

mag_info_channels

The counts for enquest are: 
- 'Premier programme': 1463,
- 'La Première': 61,
- 'Radio-Genève': 3,
- 'Deuxième programme': 3,
- 'Radio-Lausanne': 1

Chosen --> 'Premier programme'
Should we set fixed channels per alias??

- programs with "radio-geneve" and "Radio-Lausanne" are indeed pre-1974 so it's consistent,  but are probably errors?
- programs with "Deuxième programme" are only 2 other apart from the one with multiple channels --> probably an error

-> Conclusions --> choose 'Premier programme' and map "Deuxième programme", ("radio-geneve" and "Radio-Lausanne"?) to 'Premier programme'

#### c. `min_oecu`: 'La Première', 'Espace 2'

In [ ]:
# min_oecu
min_oecu_df = rts_metadata_df[rts_metadata_df['alias']=='min_oecu']

print(min_oecu_df['radio_channels'].values)
min_oecu_channels = Counter([t for v in min_oecu_df['radio_channels'].values for t in v ])

min_oecu_channels

The counts for enquest are: 
- 'La Première': 193, 
- 'Premier programme': 138, 
- 'Espace 2': 1

Chosen --> 'La Première'

#### d. `miroir_monde`: 'Radio Sottens', 'Premier programme'

In [ ]:
# min_oecu
miroir_monde_df = rts_metadata_df[rts_metadata_df['alias']=='miroir_monde']

print(miroir_monde_df['radio_channels'].values)
miroir_monde_channels = Counter([t for v in miroir_monde_df['radio_channels'].values for t in v ])

miroir_monde_channels

The counts for enquest are: 
- 'Premier programme': 2323,
- 'Radio-Genève': 38,
- 'Radio-Lausanne': 30,
- 'Radio Sottens': 22,
- 'Deuxième programme': 4,
- 'La Première': 3

Chosen --> 'Premier programme'

- programs with "Deuxième programme" are only 2 other apart from the one with multiple channels --> probably an error

- radio sottens, radio-Lausanne and radio-geneve are probably also errors but had to know for sure here

#### e. `petitdej`: 'Option Musique', 'La Première'

In [ ]:
# min_oecu
petitdej_df = rts_metadata_df[rts_metadata_df['alias']=='petitdej']

print(petitdej_df['radio_channels'].values)
petitdej_channels = Counter([t for v in petitdej_df['radio_channels'].values for t in v ])

petitdej_channels

The counts for enquest are: 
- 'La Première': 1582, 
- 'Option Musique': 1

Chosen --> 'La Première'

#### f. `petitsuisse_eurodej`: 'Premier programme', 'Deuxième programme'

In [ ]:
# min_oecu
suisse_euro_df = rts_metadata_df[rts_metadata_df['alias']=='suisse_euro']

print(suisse_euro_df['radio_channels'].values)
suisse_euro_channels = Counter([t for v in suisse_euro_df['radio_channels'].values for t in v ])

suisse_euro_channels

The counts for enquest are: 
- 'Premier programme': 131,
- 'Deuxième programme': 131

It's literally 50/50 so there is no actual way of knowing.
Chosen --> 'Premier programme'

### Now unify the radio_channel values

We will choose systematically "premier programme" or "la première" when we have a choice.

Another modification might be to change all occurences of "Premier programme" to 'La Première' and of 'Deuxième programme' to "Espace 2"

In [112]:
# Define the priority for types
priority_order = {
    'La Première': 1,
    'Premier programme': 1,
    'Deuxième programme': 5,
    'Espace 2': 5,
    'Radio Sottens': 10,
    'Radio-Genève': 10,
    'Radio-Lausanne': 10,
    'Option Musique': 10
}

# Apply to normalized_types column - keeps only the highest priority (lowest number)
"""rts_metadata_df['radio_channel'] = rts_metadata_df['radio_channels'].apply(
    lambda x: None if len(x)==0 else min(x, key=lambda t: priority_order.get(t, 999))
)"""

rts_metadata_df['radio_channel'] = rts_metadata_df['radio_channels'].apply(lambda x: x[0] if len(x)==1 else (None if x==[] else min(x, key=lambda t: priority_order.get(t, 999))))

print(any(rts_metadata_df['radio_channel'].isna()))
multiple_channels = rts_metadata_df['radio_channel'].apply(lambda x: isinstance(x, list))
print(any(multiple_channels))

rts_metadata_df

True
False


,alias,date_str,stt_filename,mp3_filenames,stripped_OID,exact_date,broadcast_date,cls_ID,OID,login,...,subdomains,recording_dates,first_broadcast_dates,supports,spt_filenames,participants,broadcast_types,broadcast_types_en,normalized_types,radio_channel
0,ana_media,16/12/1996,677E6735-8DEA-44F1-A52F-142BFEF9AB2E_STT.xml,['677e6735-8dea-44f1-a52f-142bfef9ab2e_7UBM_05...,677E6735-8DEA-44F1-A52F-142BFEF9AB2E,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{677E6735-8DEA-44F1-A52F-142BFEF9AB2E},NaN,...,['Interview'],['~__/12/1996 - ~__/12/1996'],['16/12/1996 - 16/12/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, '7UBM_057338_1{8cfcf300-a815-4f92-bea2-...","[{'name': 'Frias, Roxanne', 'function': 'Inter...",[Interview],[Interview],discussion,Espace 2
1,ana_media,26/05/1997,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484_STT.xml,['640e3c0b-40f0-4bd9-83c6-1ff4bce16484_1BBM_05...,640E3C0B-40F0-4BD9-83C6-1FF4BCE16484,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{640E3C0B-40F0-4BD9-83C6-1FF4BCE16484},NaN,...,"['Commentaire', 'Parlé divers']",['~__/05/1997 - ~__/05/1997'],['26/05/1997 - 26/05/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['1BBM_058066_6{2782ba11-7050-44d3-bef1-a81252...,"[{'name': 'Dubois, Laurent', 'function': 'Inte...","[Commentaire, Parlé divers]","[Commentary, Miscellaneous speech]",article,Espace 2
2,ana_media,20/01/1997,2AF13E50-C6A2-49E5-9851-06C441833B50_STT.xml,['2af13e50-c6a2-49e5-9851-06c441833b50_RPBM_05...,2AF13E50-C6A2-49E5-9851-06C441833B50,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{2AF13E50-C6A2-49E5-9851-06C441833B50},NaN,...,['Interview'],['~__/04/1997 - ~__/04/1997'],['20/01/1997 - 20/01/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, 'RPBM_057474_1{8116d0d3-1c12-44cf-a8a2-...","[{'name': 'Duparc, Nicole', 'function': 'Inter...",[Interview],[Interview],discussion,Espace 2
3,ana_media,11/11/1996,C940BD19-5503-4918-852B-D40C1C18B359_STT.xml,['c940bd19-5503-4918-852b-d40c1c18b359_O1BM_05...,C940BD19-5503-4918-852B-D40C1C18B359,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{C940BD19-5503-4918-852B-D40C1C18B359},NaN,...,['Interview'],['~__/11/1996 - ~__/11/1996'],['11/11/1996 - 11/11/1996'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['O1BM_057153_2{396b576c-7ed9-4018-a251-c12a84...,"[{'name': 'Kiefer, B.', 'function': 'Interview...",[Interview],[Interview],discussion,Espace 2
4,ana_media,21/04/1997,6EE0054C-042D-42E6-B11F-FEB6C0353B5A_STT.xml,['6ee0054c-042d-42e6-b11f-feb6c0353b5a_FHBM_05...,6EE0054C-042D-42E6-B11F-FEB6C0353B5A,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{6EE0054C-042D-42E6-B11F-FEB6C0353B5A},NaN,...,['Interview'],['~__/04/1997 - ~__/04/1997'],['21/04/1997 - 21/04/1997'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,['FHBM_055791_2{e8455981-078d-44f2-bb94-2e22cd...,"[{'name': 'Prophan, Geneviève', 'function': 'I...",[Interview],[Interview],discussion,Espace 2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26256,vie_va,12/11/1984,B36F7349-DA77-412C-A42D-F8C7A5FF8E61_STT.xml,['b36f7349-da77-412c-a42d-f8c7a5ff8e61_34180.0...,B36F7349-DA77-412C-A42D-F8C7A5FF8E61,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{B36F7349-DA77-412C-A42D-F8C7A5FF8E61},NaN,...,['Interview'],['12/11/1984 - 12/11/1984 / Avant'],['12/11/1984 - 12/11/1984'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, '34180.01{4A2F552B-CB07-4FA7-B533-B927B...","[{'name': 'Pache, Jean', 'function': 'Intervie...",[Interview],[Interview],discussion,Espace 2
26257,vie_va,07/11/1984,DC22DFF9-49DA-42C0-909B-55B9E4B78D33_STT.xml,['dc22dff9-49da-42c0-909b-55b9e4b78d33_34178{7...,DC22DFF9-49DA-42C0-909B-55B9E4B78D33,True,True,{D2593F4E-C887-4E48-8982-5BD08BA4DAE0},{DC22DFF9-49DA-42C0-909B-55B9E4B78D33},NaN,...,['Interview'],['07/11/1984 - 07/11/1984 / Avant'],['07/11/1984 - __/__/____'],[{'spt_clsid': '{9C19C69A-A4DC-479E-94F1-61FF7...,"[None, '34178{7E2BE637-35EC-4E3D-B65C-CBF9973E...","[{'name': 'Pool, Lça', 'function': 'In

We will see in the future if we want to do further processing of the radio channels

In [42]:
suisse_euro_df.sort_values('date_str', axis=0)[['alias', 'date_str', 'radio_channels', 'recording_place']]
# programs with raqdio-geneve are indeed pre-1974 so it's consistent

,alias,date_str,radio_channels,recording_place
25165,suisse_euro,01/04/1972,[Premier programme],NaN
24922,suisse_euro,01/05/1971,[Premier programme],NaN
24997,suisse_euro,01/06/1974,[Premier programme],Lausanne (Studio RSR)
25040,suisse_euro,01/06/1977,[Deuxième programme],NaN
25181,suisse_euro,01/07/1972,[],Lausanne (Studio RSR)
...,...,...,...,...
25121,suisse_euro,30/11/1977,[Deuxième programme],NaN
25126,suisse_euro,31/03/1973,[Premier programme],NaN
25122,suisse_euro,31/03/1976,[Deuxième programme],NaN
25172,suisse_euro,31/08/1974,[Premier programme],Lausanne (Studio RSR)


### Clean the rest of the list-valued columns by literally evaluating them into their intended data structure

The columns in question are:
- participants
- geographical_descriptors
- person_descriptors
- thematical_descriptors
- recording_dates
- first_broadcast_dates

In [113]:
def literal_eval_or_none(x):
    if isinstance(x, str):
        return literal_eval(x) 
    elif isinstance(x, list) and len(x)!=0:
        return x
    else: 
        return []

In [114]:
rts_metadata_df['participants'] = rts_metadata_df['participants'].apply(literal_eval_or_none)
rts_metadata_df['geographical_descriptors'] = rts_metadata_df['geographical_descriptors'].apply(literal_eval_or_none)
rts_metadata_df['person_descriptors'] = rts_metadata_df['person_descriptors'].apply(literal_eval_or_none)
rts_metadata_df['thematical_descriptors'] = rts_metadata_df['thematical_descriptors'].apply(literal_eval_or_none)
rts_metadata_df['recording_dates'] = rts_metadata_df['recording_dates'].apply(literal_eval_or_none)
rts_metadata_df['first_broadcast_dates'] = rts_metadata_df['first_broadcast_dates'].apply(literal_eval_or_none)

In [115]:
rts_metadata_df['participants'][0], type(rts_metadata_df['participants'][0]), type(rts_metadata_df['participants'][0][0])

([{'name': 'Frias, Roxanne',
   'function': 'Interviewé/e',
   'role': 'journaliste à M6'},
  {'name': 'Dirren, Sarah',
   'function': 'Intervieweur/euse',
   'role': 'collaboratrice RSR'}],
 list,
 dict)

## Format data into the final dict, with keys and issue_id as keys

In [116]:
# use the issue_index as base to ease things:
rts_issue_index_path = "../text_preparation/data/issue_indices/issue_index.rts.json"

with open(rts_issue_index_path, "r", encoding='utf-8') as f:
    rts_issue_idx = json.load(f)

#### Final cleaning: Removing dateless issues and sorting by date

In [117]:
# now only write the issues which have a date
final_rts_metadata_df = rts_metadata_df[~rts_metadata_df['date_str'].isna()]

# sort them by date
final_rts_metadata_df['date_dt'] = pd.to_datetime(final_rts_metadata_df['date_str'], format="%d/%m/%Y")
final_rts_metadata_df = final_rts_metadata_df.sort_values(by=['alias', 'date_dt'])

print(f"There are {len(final_rts_metadata_df)} non-duplicated records.")
rts_metadata_dict = final_rts_metadata_df.to_dict('index')
len(final_rts_metadata_df)

There are 26161 non-duplicated records.


26161

### Writing to dict format

In [118]:
all_rts_issues =  {alias: {} for alias in rts_metadata_df['alias'].drop_duplicates().values}

for idx, row_dict in rts_metadata_dict.items():

    day, month, year = row_dict['date_str'].split('/')

    alias = row_dict['alias']

    issue = [i for i in rts_issue_idx[alias][year][month] if i['day']==day and i['stripped_OID']==row_dict['stripped_OID']]

    if len(issue)!=1:
        msg = f"WARNING! {row_dict['alias']}-{row_dict['date_str']}, WE HAVE {len(issue)} ISSUES WITH THIS OID!!"
        print(msg)
    else:
        # get issue element
        issue = issue[0]

    issue_id = f"{alias}-{year}-{month}-{day}-{issue['edition']}"

    if not isinstance(row_dict['geographical_descriptors'], list):
        print(f"row_dict['geographical_descriptors'] is not a list! maybe we need another formatting")

    all_rts_issues[alias][issue_id] = {
        "processed_metadata": {
            "exact_date": row_dict['exact_date'],
            "broadcast_date": row_dict['broadcast_date'],
            "radio_channel": row_dict['radio_channel'],
            "broadcast_type": row_dict['normalized_types'],
        },
        "partner_provided_metadata": {
            "OID": row_dict['OID'],
            "broadcast_episode_title": row_dict['broadcast_episode_title'],
            "broadcast_program_name": row_dict['broadcast_program_name'],
            "hierarchy_level": row_dict['hierarchy_level'],
            "physical_support_history": row_dict['physical_support_history'],
            "production_type": row_dict['production_type'],
            "recording_place": row_dict['recording_place'],
            "series_title": row_dict['series_title'],
            "content_summary": row_dict['content_summary'],
            "live": row_dict['live'],
            "modulation_type": row_dict['modulation_type'],
            "work_duration": row_dict['work_duration'],
            "geographical_descriptors": row_dict['exact_date'],
            "person_descriptors": row_dict['person_descriptors'],
            "thematical_descriptors": row_dict['thematical_descriptors'],
            "recording_dates": row_dict['first_broadcast_dates'],
            "participants": row_dict['participants'],
            "broadcast_types_en": row_dict['broadcast_types_en'],
            "radio_channels": row_dict['radio_channels'],
        }
    }

In [ ]:
all_rts_issues

In [120]:
OUTPUT_METADATA_FILE = "/home/piconti/impresso-text-acquisition/text_preparation/data/sample_data/RTS/issues_metadata.rts.json"

In [121]:

with open(OUTPUT_METADATA_FILE, "w") as fout:
    json.dump(all_rts_issues, fout, indent=4)